# Panorama Image Stitching
## Midterm Report — Proposal and Sequential Baseline

**CSC14120 · Parallel Programming · Group 16 · V0**

Notebook trình bày phương pháp của nhóm, cài đặt tuần tự, kiểm chứng hình học, panorama và phân tích hiệu năng trước khi chuyển sang CUDA. Bảng kết quả và hình được sinh từ mã nguồn bài nộp trong phiên đang chạy.

Bản nộp lưu đầy đủ output. Bản quản lý trong Git giữ source cell nhưng không lưu output. Timing có thể dao động khi chạy lại; các nhận xét định lượng trong bản nộp được đối chiếu với output đã lưu của chính bản đó.

### Contents

- [0. Reproducibility Setup](#setup)
- [3.1 Team Information and Work Plan](#team)
- [3.2 Problem Understanding](#problem)
- [3.3 Chosen Methods](#methods)
- [3.4 Sequential Implementation](#implementation)
- [3.5 Results and Profiling](#results)
- [3.6 Dataset](#dataset)
- [3.7 GPU Execution Plan](#gpu-plan)
- [References](#references)
- [Reproducibility Checklist](#checklist)

<a id="setup"></a>
## 0. Reproducibility Setup
### 0.1 Runtime Verification

Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy. V0 thực thi trên CPU của phiên Colab; T4 là môi trường build/test theo quy định môn học. Tên CPU được lưu để diễn giải kết quả V0.

In [ ]:
# Authors: Thanh Chuong (23120195)
# External source: none
from pathlib import Path
import csv
import datetime as dt
import hashlib
import io
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import uuid

def execute(command, cwd=None, timeout=1800, check=True):
    """Run one explicit command; a failure is reported without an automatic retry."""
    print("$", shlex.join([str(part) for part in command]), flush=True)
    result = subprocess.run([str(part) for part in command], cwd=cwd,
                            capture_output=True, text=True, timeout=timeout)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check:
        result.check_returncode()
    return result

gpu_output = execute(["nvidia-smi"]).stdout
gpu_name = execute(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip()
if not re.search(r"\bT4\b", gpu_name):
    raise RuntimeError("Official execution requires a Tesla T4 Colab runtime.")
compiler_version = execute(["nvcc", "--version"]).stdout
cpu_lines = Path("/proc/cpuinfo").read_text().splitlines()
cpu_name = next((line.split(":", 1)[1].strip() for line in cpu_lines
                 if line.startswith("model name")), platform.processor())
SESSION_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
ENVIRONMENT = {"session_id": SESSION_ID, "started_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
               "gpu": gpu_name, "cpu": cpu_name, "python": sys.version,
               "platform": platform.platform(), "nvcc": compiler_version}
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))

### 0.2 Input Configuration

Notebook build chương trình từ thư mục `src/` của bài nộp. Trước khi Run all:

1. Mở panel **Files** của phiên Colab và tải `src/` lên: tạo thư mục `/content/src` rồi tải các file vào đó, hoặc tải một file `src.zip` vào `/content`. Nếu chưa có, cell bên dưới hỏi file `src.zip`.
2. Phần còn lại (dataset đã chuẩn bị, `scripts/`, `tests/` và ảnh nguồn của cặp synthetic) nằm trong file `Group16_support.zip` trên Drive chung của nhóm. Notebook tự tải file này bằng `gdown` theo `SUPPORT_DRIVE_ID`; nếu không tải được thì hỏi upload. Có thể đặt sẵn file tại `/content/Group16_support.zip`.

`src/` chỉ lấy từ bài nộp; file hỗ trợ không chứa mã nguồn chương trình. Chế độ `existing` dùng một bản checkout đầy đủ của repo tại `EXISTING_WORKSPACE`, chỉ để thử nghiệm.

Dataset chuẩn bị: [Google Drive](https://drive.google.com/drive/folders/1HOvnly1k1QHOsV17qLzSAG4ihszNzCqO?usp=sharing). Ảnh gốc: [Google Drive](https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing). File hỗ trợ: [Group16_support.zip](https://drive.google.com/file/d/1Jx6o5kUnpvxXVEHfYKD1GBvZMTbSeEGu/view?usp=sharing).

Benchmark dùng `scripts/run_benchmark.py` với các đối số `--stitch --dataset --runs --out`. `ACCELERATED_STAGES` là tập stage nhóm ưu tiên tối ưu (P3-T01), dùng cho ước lượng Amdahl ở mục 3.5.6.

In [ ]:
SOURCE_MODE = "colab"  # colab: src/ uploaded to this session; existing: full repository checkout
SRC_UPLOAD_DIR = Path("/content/src")
SRC_ARCHIVE = Path("/content/src.zip")
SUPPORT_ARCHIVE = Path("/content/Group16_support.zip")
SUPPORT_DRIVE_ID = "1Jx6o5kUnpvxXVEHfYKD1GBvZMTbSeEGu"  # Drive file id of the shared Group16_support.zip
EXISTING_WORKSPACE = "/content/panorama-stitching"
RAW_SYNTHETIC_RELATIVE = "raw/tuan/standard1_left.jpg"
EXPECTED_SYNTHETIC_SHA256 = "e9fbe7ecfed6344a36610fe50f2c6019bcd1525f4e217c033ed8ded4f90bf1e8"
PANORAMA_SIZE = "medium"
RUNS = 5
# Argument list of scripts/run_benchmark.py (P3-T04); placeholders are filled in cell 3.5.4.
BENCHMARK_ARGS = ["--stitch", "{stitch}", "--dataset", "{dataset}", "--runs", "{runs}", "--out", "{out}"]
ACCELERATED_STAGES = [2, 4, 6]  # P3-T01: stages prioritized for GPU optimization
QUALITY_REVIEW_CONFIRMED = True  # Set only after inspecting panoramas and native-resolution crops.
FINAL_REPORT_CONFIRMED = True  # Set after checking this report against the current-session evidence.
REQUIRE_COMPLETE_RUN = False  # Optional strict failure mode; READY depends on all checks regardless of this flag.

In [ ]:
# Copy the submitted src/ into this session, then add the support files
# (prepared dataset, scripts, tests and the synthetic source image).
import stat
import zipfile


def safe_extract(archive, destination):
    """Reject traversal, duplicate targets and symbolic links before writing."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as bundle:
        seen = set()
        for entry in bundle.infolist():
            target = (destination / entry.filename.replace("\\", "/")).resolve()
            if (not target.is_relative_to(destination) or target == destination or target in seen
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe or duplicate archive member: " + entry.filename)
            seen.add(target)
        for entry in bundle.infolist():
            target = destination / entry.filename.replace("\\", "/")
            if entry.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(entry) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)


def folders_with(root, marker):
    """Folders under root (root included) that contain the relative file marker."""
    return [folder for folder in [root, *root.rglob("*")]
            if folder.is_dir() and (folder / marker).is_file()]


def request_upload(message):
    from google.colab import files
    print(message)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one ZIP archive.")
    return Path(next(iter(uploaded))).resolve()


if SOURCE_MODE == "existing":
    PROJECT = Path(EXISTING_WORKSPACE).resolve()
elif SOURCE_MODE == "colab":
    PROJECT = Path("/content") / ("group16-" + SESSION_ID)
    PROJECT.mkdir()
    if (SRC_UPLOAD_DIR / "main.cu").is_file():
        shutil.copytree(SRC_UPLOAD_DIR, PROJECT / "src")
    else:
        archive = SRC_ARCHIVE if SRC_ARCHIVE.is_file() else request_upload(
            "Tải lên src.zip chứa thư mục src/ của bài nộp.")
        staging = PROJECT / "_src_archive"
        safe_extract(archive, staging)
        roots = folders_with(staging, "main.cu")
        if len(roots) != 1:
            raise ValueError("The source archive must contain exactly one folder with main.cu.")
        shutil.move(str(roots[0]), str(PROJECT / "src"))
        shutil.rmtree(staging)
    if not SUPPORT_ARCHIVE.is_file() and SUPPORT_DRIVE_ID:
        try:
            import gdown
            gdown.download(id=SUPPORT_DRIVE_ID, output=str(SUPPORT_ARCHIVE), quiet=False)
        except Exception as error:  # Fall back to a manual upload below.
            print("Cannot download the support archive:", error)
    archive = SUPPORT_ARCHIVE if SUPPORT_ARCHIVE.is_file() else request_upload(
        "Tải lên Group16_support.zip (dataset, scripts, tests, ảnh nguồn synthetic).")
    staging = PROJECT / "_support_archive"
    safe_extract(archive, staging)
    roots = folders_with(staging, "dataset/manifest.csv")
    if len(roots) != 1:
        raise ValueError("The support archive must contain exactly one dataset/manifest.csv.")
    if (roots[0] / "src").exists():
        raise ValueError("The support archive must not contain src/; src/ comes only from the submission.")
    for name in ("dataset", "scripts", "tests", "raw"):
        if (roots[0] / name).is_dir():
            shutil.move(str(roots[0] / name), str(PROJECT / name))
    shutil.rmtree(staging)
else:
    raise ValueError("SOURCE_MODE must be colab or existing.")

required = ["src/main.cu", "scripts/midterm_notebook.py", "scripts/run_benchmark.py",
            "tests/run_v0_features.py", "scripts/make_synthetic_pair.py",
            "scripts/homography_error.py", "dataset/manifest.csv", RAW_SYNTHETIC_RELATIVE]
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise FileNotFoundError("Missing workspace inputs: " + ", ".join(missing))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
RESULTS = PROJECT / "results" / "midterm" / SESSION_ID
BUILD = PROJECT / "build" / "midterm"
RESULTS.mkdir(parents=True, exist_ok=False)
BUILD.mkdir(parents=True, exist_ok=True)
print("Workspace:", PROJECT)
print("Source files:", len(list((PROJECT / "src").iterdir())))
print("Current-session evidence:", RESULTS)

### 0.3 Dependencies and Provenance

Python chỉ phục vụ chuẩn bị dữ liệu, kiểm chứng và trình bày; chương trình `src/` vẫn chỉ dùng C/C++ chuẩn và stb. Mỗi phiên lưu hash nguồn và phiên bản Python packages. Dataset và kết quả không được đưa vào Git.

In [ ]:
import importlib.metadata as package_metadata
from PIL import Image as PILImage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as DisplayImage
from midterm_notebook import (SIZES, SITES, TIMING_COLUMNS, sha256, validate_dataset,
                              parse_stitch_csv, collect_benchmark_csv,
                              source_manifest, preview_image, overlap_box)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 10})
ENVIRONMENT["packages"] = {name: package_metadata.version(name)
                           for name in ("numpy", "Pillow", "pandas", "matplotlib")}
SOURCE_HASHES = source_manifest(PROJECT)
git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT, capture_output=True, text=True)
ENVIRONMENT["git_commit"] = git.stdout.strip() if git.returncode == 0 else "unavailable (uploaded source)"
(RESULTS / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
(RESULTS / "source_manifest.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")
print("Source files fingerprinted:", len(SOURCE_HASHES))
print(json.dumps(ENVIRONMENT["packages"], indent=2))
CHECKS = {"runtime_t4": True, "source_recorded": True}

### 0.4 Dataset Integrity

Kiểm tra đủ bốn cặp, ba kích thước, RGB JPEG, không EXIF và SHA-256 trùng manifest. Sai dữ liệu phải dừng trước build/đo; không bỏ âm thầm cặp lỗi để tạo bảng thiếu.

In [ ]:
DATASET = PROJECT / "dataset"
verified_images = validate_dataset(DATASET)
display(pd.DataFrame(verified_images)[["file", "width", "height", "sha256"]])
RAW_SYNTHETIC = PROJECT / RAW_SYNTHETIC_RELATIVE
raw_digest = sha256(RAW_SYNTHETIC)
print("Synthetic source:", RAW_SYNTHETIC_RELATIVE, "SHA-256:", raw_digest)
if EXPECTED_SYNTHETIC_SHA256 and raw_digest != EXPECTED_SYNTHETIC_SHA256:
    raise ValueError("Synthetic source differs from the source documented in S5 section 5.5.")
CHECKS["dataset_verified"] = True
CHECKS["synthetic_source_verified"] = True

In [ ]:
# Recorded team metadata, not benchmark measurements.
report_context = {
    "members": {"Nguyen Ho Anh Tuan": "23120185", "Tran Huu Kim Thanh": "23120166",
                "Le Thuong De": "23120232", "Le Ha Thanh Chuong": "23120195"},
    "dataset_collection": {"registered_sites": 9, "primary_pairs": 13, "standard_pairs": 9,
                           "challenging_pairs": 4, "target_overlap_percent": [30, 50], "jpeg_quality": 95},
    "source_dimensions_and_crops": {"site1": [4096, 3072, 0, 384, 4096, 2688],
        "site2": [8160, 6144, 0, 777, 8160, 5367], "site3": [4032, 3024, 0, 378, 4032, 2646],
        "challenge": [4096, 2304, 0, 0, 4096, 2304]},
    "design": {"s1_rgb_weights": [0.299, 0.587, 0.114], "gray_divisor": 255,
        "s1_sigma_radius": [1, 3], "fast_arc_ring_radius": [9, 16, 3], "fast_threshold": 0.05,
        "nms_shape": [3, 3], "max_keypoints": 2000, "border": 15,
        "brief_bits_words_patch": [256, 8, 31], "brief_pattern_sigma": 6.2,
        "s3_sigma_radius": [2, 4], "seed": 16, "ratio": 0.8, "ratio_integer_rule": [5, 4],
        "cross_check": 0, "ransac_sample_cap_confidence": [4, 2000, 0.99],
        "ransac_tau_small_medium_large": [2, 3, 6], "homography_epsilon": 1e-6,
        "blend_weights": [0.5, 0.5], "canvas_max_dimension": 10000, "canvas_max_pixels": 100000000,
        "warp_eps_coord_pivot_rcond": [1e-6, 1e-12, 1e-12]},
    "synthetic_acceptance": {"rmse_px": 0.5, "max_px": 1.0, "grid": [20, 20]},
}
print("Thông tin Nhóm và tham số thiết kế đã chốt; không phải số đo timing:")
print(json.dumps(report_context, ensure_ascii=False, indent=2))

<a id="team"></a>
## 3.1 Team Information and Work Plan

### Team Members

| Họ tên | MSSV | Stage phụ trách ở V0 |
|---|---|---|
| Nguyễn Hồ Anh Tuấn (nhóm trưởng) | 23120185 | S1 Preprocessing |
| Trần Hữu Kim Thành | 23120166 | S2 Keypoint detection, S3 Description |
| Lê Thượng Đế | 23120232 | S4 Matching, S5 Homography |
| Lê Hà Thanh Chương | 23120195 | S6 Warping and blending |

Mỗi stage có một người phụ trách. Người đó chọn phương pháp và tham số, viết tài liệu thiết kế trong `docs/stages/`, cài đặt và kiểm thử stage của mình. Các stage giao tiếp qua interface chung trong `src/common.h`, nên từng stage được kiểm thử độc lập bằng dữ liệu tự tạo trước khi ghép thành pipeline.

### Version Responsibilities

| Thành viên | V0 tuần tự | V1 Basic CUDA | V2, V3 |
|---|---|---|---|
| Tuấn | S1 | Kernel S1 | Tối ưu S1 nếu profiling V1 cho thấy S1 đáng kể |
| Thành | S2, S3 | Kernel S2, S3 | Tối ưu S2 (stage ưu tiên), S3 theo profiling |
| Đế | S4, S5 | Kernel S4, S5 | Tối ưu S4 (stage ưu tiên), S5 theo profiling |
| Chương | S6 | Kernel S6 | Tối ưu S6 (stage ưu tiên) |

S2, S4 và S6 là ba stage chiếm nhiều thời gian nhất của V0 ở cả ba độ phân giải (mục 3.5), nên được ưu tiên khi tối ưu ở V2 và V3. Danh sách tối ưu cụ thể của V2, V3 chỉ chốt sau khi có profiling V1. Mỗi người giữ đúng phương pháp và tham số của stage mình từ V0 đến V3 để speedup so sánh được.

### Midterm Work Plan

Nhóm chia việc giữa kỳ thành năm giai đoạn: chuẩn bị (chụp ảnh, chốt interface), cài đặt từng stage, ghép và kiểm chứng V0, đo và phân tích, hoàn thiện báo cáo. Công việc được theo dõi bằng bảng task trong repo GitHub riêng của nhóm. Code của mỗi stage đi qua pull request trước khi vào nhánh `main`; mọi lần build, test và đo chính thức chạy trên Google Colab với GPU Tesla T4.

<a id="problem"></a>
## 3.2 Problem Understanding

### Problem

Đầu vào là hai ảnh chụp cùng một cảnh, có một phần nội dung chung. Ta gọi ảnh trái là A, ảnh phải là B. Đầu ra là một ảnh panorama duy nhất chứa cả hai ảnh, trong đó vùng chung của A và B nằm chồng khít lên nhau.

Hai ảnh được chụp từ cùng một chỗ đứng, người chụp chỉ xoay máy. Khi tâm máy ảnh không dời đi, mọi điểm của ảnh B liên hệ với điểm tương ứng trong ảnh A qua một phép biến đổi phối cảnh (homography) H kích thước 3×3. Vì vậy bài toán ghép ảnh quy về việc tìm H rồi dùng H để đưa B vào hệ tọa độ của A. Nếu người chụp bước sang ngang, vật ở gần và vật ở xa dịch đi những khoảng khác nhau (parallax). Khi đó không có H nào khớp cho toàn ảnh và panorama sẽ bị lệch hoặc có ảnh kép. Đây là lý do đề yêu cầu "xoay máy, không bước đi" và chồng lấn khoảng 30–50%.

H có 8 bậc tự do, nên chỉ cần 4 cặp điểm tương ứng không có 3 điểm nào thẳng hàng là giải được. Khó khăn nằm ở chỗ chương trình phải tự tìm các cặp điểm đó trong hàng triệu pixel, và một phần các cặp tìm được sẽ sai.

### Six-Stage Pipeline

```mermaid
flowchart LR
    A["Ảnh A"] --> S1A["S1: ảnh xám, làm mượt"]
    B["Ảnh B"] --> S1B["S1: ảnh xám, làm mượt"]
    S1A --> S2A["S2: keypoint"] --> S3A["S3: descriptor"]
    S1B --> S2B["S2: keypoint"] --> S3B["S3: descriptor"]
    S3A --> S4["S4: ghép descriptor"]
    S3B --> S4
    S4 --> S5["S5: ước lượng H bằng RANSAC"]
    S5 --> S6["S6: warp B vào hệ A, trộn vùng chung"]
    S6 --> P["Panorama"]
```

1. Ở S1 (tiền xử lý), ảnh màu được đổi sang ảnh xám vì các bước sau chỉ cần độ sáng. Sau đó ảnh được làm mượt bằng bộ lọc Gaussian để bớt nhiễu. Nếu không làm mượt, nhiễu từng pixel dễ bị nhận nhầm là góc ở S2.
2. S2 tìm keypoint, tức những điểm dễ nhận ra lại trong ảnh kia, thường là góc, nơi độ sáng thay đổi theo nhiều hướng. Điểm trên vùng trơn hoặc dọc một cạnh thẳng thì không dùng được, vì có nhiều vị trí lân cận trông giống hệt nhau.
3. S3 gắn cho mỗi keypoint một descriptor, là một dãy số tóm tắt vùng ảnh nhỏ quanh nó. Hai keypoint nhìn cùng một chi tiết của cảnh sẽ có descriptor gần nhau.
4. S4 ghép cặp: với mỗi descriptor của A, chương trình tìm descriptor gần nhất trong B. Cặp nào mơ hồ, tức descriptor gần nhất và gần nhì trong B gần như bằng nhau, thì bị loại, vì vùng lặp lại như ô gạch hay cửa sổ dễ sinh ra cặp sai.
5. S5 ước lượng homography. Danh sách cặp sau S4 vẫn còn cặp sai, nên không thể giải H trực tiếp từ tất cả các cặp. RANSAC lặp lại nhiều lần: chọn ngẫu nhiên 4 cặp, giải ra một H, rồi xem bao nhiêu cặp khác khớp với H đó (inlier) và sai số của chúng lớn đến đâu. H có điểm tốt nhất được giữ lại và tính lại từ toàn bộ inlier của nó.
6. S6 warp và trộn ảnh. Từ H và kích thước hai ảnh, chương trình tính khung panorama đủ chứa cả A và B đã biến đổi. Với mỗi pixel của khung, chương trình tính ngược ra vị trí tương ứng trong A và trong B, lấy màu bằng nội suy, rồi trộn hai màu ở vùng chồng lấn.

S1 đến S3 chạy riêng trên từng ảnh và không phụ thuộc ảnh kia. S4 đến S6 cần dữ liệu của cả hai ảnh.

### Data Contracts

| Ranh giới | Dữ liệu |
|---|---|
| Vào S1 | Ảnh RGB 8 bit, đọc bằng stb |
| S1 → S2, S3 | Ảnh xám `float` trong [0, 1] |
| S2 → S3 | Danh sách keypoint: tọa độ pixel, điểm số |
| S3 → S4 | Descriptor nhị phân 256 bit cho mỗi keypoint |
| S4 → S5 | Danh sách cặp chỉ số (keypoint A, keypoint B) kèm khoảng cách Hamming |
| S5 → S6 | Ma trận H 3×3 biến điểm của B thành điểm của A |
| Ra S6 | Ảnh RGB panorama và độ lệch gốc tọa độ của A trong panorama |

Chi tiết bố cục bộ nhớ và mã lỗi nằm trong `docs/interface.md`.

### Evaluation

Một panorama được coi là tốt khi vùng chồng lấn khớp, không có cạnh bị đứt hay ảnh kép. Nhìn bằng mắt là chưa đủ để so sánh, nên Nhóm còn dùng một cặp ảnh synthetic: lấy một ảnh thật làm cảnh, tự tạo ảnh B từ ảnh A bằng một H biết trước, chạy pipeline rồi so H tìm được với H thật. Sai số được đo bằng khoảng cách (pixel) giữa vị trí mà hai ma trận chiếu cùng một lưới điểm của B vào A.

Về thời gian, đề yêu cầu đo riêng từng stage ở ba kích thước 1280×720, 1920×1080 và 3840×2160. Từ đó Nhóm xác định stage nào tốn thời gian nhất và ước lượng speedup tối đa theo định luật Amdahl trước khi chuyển sang CUDA.

<a id="methods"></a>
## 3.3 Chosen Methods

### 3.3.1 S1 Preprocessing

#### Method and Parameters

S1 đổi ảnh màu RGB sang ảnh xám theo hệ số BT.601, sau đó làm mượt bằng Gaussian blur separable: một lượt ngang, một lượt dọc. Pixel ngoài biên lấy bằng pixel biên gần nhất (clamp). Mỗi ảnh của cặp được xử lý độc lập.

| Tham số | Giá trị |
|---|---|
| Ảnh xám | $Y = (0.299R + 0.587G + 0.114B)/255$, kiểu `float` trong $[0, 1]$ |
| `gaussian_sigma` | 1.0 pixel |
| `gaussian_radius` | 3, tức kernel 7 tap |
| Trọng số | Tính bằng `double`, chuẩn hóa tổng bằng 1, ép về `float` |
| Biên | Clamp |
| Image pyramid | Không dùng |

Cùng một bộ tham số được dùng cho cả ba độ phân giải và cho mọi phiên bản từ V0 tới V3.

#### Algorithm and Cost

```text
for each pixel (x, y):  gray(x, y) = (0.299 R + 0.587 G + 0.114 B) / 255
for each pixel (x, y):  temp(x, y) = sum_{k=-r..r} w[k] * gray(clamp(x + k), y)
for each pixel (x, y):  out(x, y)  = sum_{k=-r..r} w[k] * temp(x, clamp(y + k))
```

Với ảnh $W \times H$ và bán kính $r$, chi phí là $O(W H (2r+1))$ phép nhân cộng cho mỗi lượt blur, bộ nhớ thêm một buffer `float` cỡ $W H$. Mỗi pixel cộng các tap theo thứ tự cố định từ $k=-r$ đến $k=r$, nên chạy lại cho kết quả giống từng bit.

#### Rationale

Ảnh xám đủ cho FAST ở S2 và BRIEF ở S3, vì hai stage này chỉ dùng độ sáng. BT.601 giữ được cạnh giữa hai vùng khác màu nhưng có cùng tổng $R+G+B$, trong khi trung bình cộng ba kênh làm mất cạnh đó. Chênh lệch chi phí giữa hai cách không đáng kể vì S1 bị giới hạn bởi băng thông bộ nhớ.

Blur loại bớt nhiễu từng pixel, vốn dễ bị FAST nhận nhầm thành góc. Gaussian 2D kích thước $k \times k$ tốn $k^2$ phép nhân cộng mỗi pixel; bản separable cho cùng kết quả (chỉ khác ở làm tròn số thực) với $2k$ phép. σ = 1 được chọn nhỏ vì FAST xét vòng tròn bán kính 3 pixel, blur mạnh hơn sẽ làm mất góc. Pyramid nhiều tỷ lệ không dùng vì hai ảnh của cặp cùng độ phân giải và gần cùng tỷ lệ; thêm pyramid sẽ kéo theo đổi interface của S2, S3.

Clamp không làm tối mép ảnh như zero padding, nên không sinh keypoint giả ở mép. Trên GPU, clamp chỉ cần `min`/`max` chỉ số.

#### GPU Suitability

Cả ba lượt đều tính độc lập trên từng pixel. V1 dự kiến dùng một thread cho một pixel output ở mỗi lượt, trọng số kernel đặt trong constant memory, ảnh xám giữ trên Device cho S2. Hai lượt blur là stencil 1D, có thể chuyển sang shared memory tile có halo hoặc gộp với bước đổi ảnh xám ở V2/V3. Mức ưu tiên được đối chiếu với timing ở mục 3.5. Mapping chi tiết và rủi ro nằm ở [mục 6 tài liệu S1](#gpu-plan).

### 3.3.2 S2 Detection

#### Method and Parameters

S2 dùng FAST-9 trên ảnh xám đã làm mượt của S1: một pixel là góc khi có ít nhất 9 pixel liên tiếp trên vòng tròn 16 điểm bán kính 3 cùng sáng hơn hoặc cùng tối hơn tâm một ngưỡng $t$. Mỗi góc được chấm bằng score V của Rosten và Drummond, sau đó qua non-maximum suppression (NMS) 3×3 và giữ N điểm mạnh nhất. S2 chỉ làm ở một tỷ lệ, không dùng image pyramid.

| Tham số | Giá trị |
|---|---|
| Số pixel liên tiếp | 9 trên vòng 16 điểm, bán kính 3 |
| `fast_threshold` $t$ | 0.05 trên thang $[0, 1]$, tức 12.75/255 |
| Score | $V = \max\left(\sum_{\text{sáng}} (d_i - t),\ \sum_{\text{tối}} (-d_i - t)\right)$ với $d_i = I_i - I_p$ |
| NMS | 3×3; hai điểm bằng score thì điểm đứng trước theo thứ tự raster thắng |
| `max_keypoints` N | 2000 mỗi ảnh |
| Border | 15 pixel, đủ cho vòng FAST và patch 31×31 của S3 |
| Thứ tự output | Score giảm dần, bằng nhau thì raster index $y \cdot W + x$ tăng dần |

#### Algorithm and Cost

```text
for each pixel p in [15, W-16] x [15, H-16]:
    d_i = I(p + offset_i) - I(p), i = 0..15
    if 9 consecutive d_i > t or 9 consecutive d_i < -t:
        score(p) = V
for each p with score(p) > 0:
    keep p if no 8-neighbour has a larger score (ties: earlier raster wins)
sort kept points by (score desc, raster asc); output the first N
```

Segment test và NMS tốn $O(W H)$ với hằng số 16; bước sắp xếp tốn $O(M \log M)$ với $M$ là số điểm còn lại sau NMS. Bộ nhớ thêm một bản đồ score `float` cỡ $W H$. Khóa (score, raster) không trùng nhau, nên thứ tự output là duy nhất.

#### Rationale

Ảnh trái và phải của một cặp được chụp đứng yên, xoay người, nên cùng độ phân giải và gần cùng tỷ lệ; bất biến tỷ lệ chưa cần. Trong các phương án đã xét (FAST-12, Harris, Shi-Tomasi, FAST xếp hạng bằng Harris như ORB, FAST nhiều tỷ lệ), FAST-9 rẻ nhất và Rosten và cộng sự (2010) cho thấy $n = 9$ lặp lại tốt hơn các giá trị $n$ khác. Score V có sẵn từ segment test nên không cần thêm bước Harris.

Ngưỡng 0.05 nằm giữa hai giá trị mặc định của OpenCV (10/255 cho `FastFeatureDetector`, 20/255 cho ORB), vì ảnh đã qua blur σ = 1 nên tương phản ở góc thấp hơn ảnh gốc. N = 2000 giữ S4 ở mức $2000 \times 2000$ phép so sánh mỗi cặp mà vùng chồng lấn vẫn còn đủ keypoint. Kết quả thực tế của các cặp được trình bày trong mục 3.5.

Một bộ tham số dùng cho cả ba kích thước. Hệ quả là ở 3840×2160, 2000 keypoint chỉ là phần nhỏ của các góc tìm được và tập trung ở các góc tương phản mạnh nhất.

#### GPU Suitability

Segment test và NMS độc lập trên từng pixel, nên V1 dùng một thread cho một pixel ở mỗi bước. Ứng viên sau NMS được ghi bằng bộ đếm atomic; thứ tự ghi không quan trọng vì bước sau sắp xếp theo khóa duy nhất (score, raster), nên output giống V0. Bước top-N cần sort trên Device. Mức ưu tiên tối ưu S2 dựa trên timing ở mục 3.5. Mapping chi tiết, hướng tối ưu và rủi ro nằm ở [mục 6 tài liệu S2](#gpu-plan).

### 3.3.3 S3 Description

#### Method and Parameters

S3 tính descriptor BRIEF 256 bit, không xoay, cho mỗi keypoint của S2. Mỗi bit là kết quả so sánh độ sáng của một cặp điểm cố định trong patch 31×31 quanh keypoint, trên một bản sao ảnh xám được làm mượt thêm bằng Gaussian σ = 2.

| Tham số | Giá trị |
|---|---|
| Số bit | 256, đóng gói thành 8 word `uint32_t` |
| Patch | 31×31, offset trong $[-15, 15]$ |
| Pattern | Kiểu G II của BRIEF: hai điểm độc lập theo Gaussian đẳng hướng σ = 31/5 = 6.2, làm tròn và kẹp trong patch |
| Sinh pattern | SplitMix64 với seed 16, sinh một lần bằng `tests/tool_gen_brief_pattern.py` và lưu thành bảng hằng `src/brief_pattern.cpp` |
| Làm mượt | Gaussian separable σ = 2, bán kính 4 (9 tap), clamp biên, dùng chung code blur với S1 |
| Bit $k$ | Nằm ở `data[k / 32]`, bit `k % 32`; bằng 1 khi điểm thứ nhất tối hơn điểm thứ hai (so sánh chặt) |
| Orientation | Không dùng |

#### Algorithm and Cost

```text
smoothed = Gaussian(sigma = 2, r = 4) of a copy of the S1 image
for each keypoint i in input order:
    (cx, cy) = (floor(x_i + 0.5), floor(y_i + 0.5))
    if the 31x31 patch leaves the image: drop keypoint i
    for k = 0..255:
        (ax, ay, bx, by) = pattern[k]
        bit k = smoothed(cx+ax, cy+ay) < smoothed(cx+bx, cy+by)
```

Làm mượt tốn $O(18 W H)$ phép nhân cộng, phần descriptor tốn $O(256 N)$ phép so sánh với $N$ keypoint. Bộ nhớ thêm hai buffer `float` cỡ $W H$. Keypoint bị bỏ được loại cùng lúc khỏi cả hai mảng, nên `descriptors[i]` luôn ứng với `keypoints[i]`. Với keypoint của S2 (border 15), không keypoint nào bị bỏ.

#### Rationale

Hai ảnh của cặp được chụp với máy giữ ngang, chỉ xoay quanh trục dọc, nên gần như không có phép xoay trong mặt phẳng ảnh. Vì vậy S3 không tính orientation như rotated BRIEF của ORB; orientation chỉ tốn thêm chi phí mà không giúp gì cho dataset này. Descriptor nhị phân 256 bit khớp kiểu `Descriptor` của interface chung và cho phép S4 ghép bằng khoảng cách Hamming. Descriptor số thực kiểu SIFT bền hơn với tỷ lệ và ánh sáng nhưng phải đổi S4 sang khoảng cách L2 và đổi interface.

BRIEF so sánh từng cặp pixel nên rất nhạy với nhiễu. Calonder và cộng sự (2010) làm mượt patch bằng Gaussian σ = 2 trên cửa sổ 9×9 trước khi so sánh; S1 chỉ blur σ = 1 để giữ góc cho FAST, nên S3 tự làm mượt thêm trên bản sao riêng. Pattern G II được bài báo BRIEF chọn vì cho kết quả tốt hơn các cách lấy mẫu ngẫu nhiên khác trong phần lớn thí nghiệm. Nhóm tự sinh pattern thay vì dùng bảng học sẵn của OpenCV để không phụ thuộc dữ liệu bên ngoài và giải thích được từng bước sinh.

Kiểm chứng descriptor bằng cặp dịch biết trước và kết quả toàn pipeline nằm ở mục 3.5.

#### GPU Suitability

Bước làm mượt dùng lại hai kernel blur của S1. Các descriptor độc lập với nhau, nên V1 dùng một thread cho một keypoint; bảng pattern 1 KB đặt trong constant memory. Bước bỏ keypoint ngoài biên dùng prefix sum để giữ đúng thứ tự. Kết quả chỉ gồm phép so sánh, nên nếu blur trên GPU giống V0 từng bit thì descriptor cũng giống từng bit. Phần làm mượt toàn ảnh cần được đối chiếu với timing ở mục 3.5 trước khi tối ưu sâu. Mapping chi tiết nằm ở [mục 6 tài liệu S3](#gpu-plan).

### 3.3.4 S4 Matching

#### Method and Parameters

S4 dùng **exact brute-force 2-NN với khoảng cách Hamming trên descriptor 256 bit**, tìm từ A sang B. Với mỗi descriptor A, chương trình xét mọi descriptor B, giữ hai láng giềng khác indexB rồi áp ratio test. S4 không dùng cross-check; các cặp còn sai hình học được xử lý ở S5.

| Thành phần | Giá trị giữ cho mọi version |
|---|---|
| Descriptor | BRIEF binary 256 bit, tám word `uint32_t` |
| Khoảng cách | Tổng số bit 1 của tám word XOR, từ 0 tới 256 |
| Số láng giềng | Hai indexB khác nhau |
| Ratio | 0.80; nhận khi $5d_1<4d_2$ |
| Cross-check | Tắt (`match_cross_check=0`) |
| Tie-break | Distance nhỏ hơn, rồi indexB nhỏ hơn |
| Output | Tăng theo indexA; nhiều A được phép chọn cùng B |

Hai láng giềng có distance bằng nhau vẫn được giữ để ratio test nhận ra sự mơ hồ. Khi $d_1=d_2=0$ hoặc $5d_1=4d_2$, cặp bị loại. So sánh nguyên 4/5 tránh việc biểu diễn `0.80f` làm sai trường hợp đúng ranh giới.

#### Algorithm and Cost

V0 chạy đơn luồng, tính Hamming bằng vòng xóa bit 1 thấp nhất `bits &= bits - 1`; không dùng SIMD viết tay. Mỗi A duy trì best và second-best trong lúc quét B, không lưu ma trận distance.

```text
for i in indices(A):
    best, second = invalid
    for j in indices(B):
        d = scalar_hamming(A[i], B[j])
        merge (d,j) into best,second using (distance,indexB)
    if 5*best.distance < 4*second.distance:
        append Match(i,best.index,best.distance)
```

Với $n_A$/$n_B$ descriptor, có $n_A n_B$ cặp và tám word/cặp, nên chi phí $O(8n_A n_B)$ với word cố định. Scratch top-two cho một A là $O(1)$; output tối đa $O(n_A)$. S4 trả lỗi khi không đủ ứng viên hoặc còn dưới bốn match, không bàn giao output một phần sang S5. Index output trỏ vào keypoint/descriptor sau compact của S3.

#### Rationale and Limitations

Hamming phù hợp với descriptor binary mà S3 cung cấp. Exhaustive search bảo đảm hai láng giềng đúng theo metric, dễ kiểm chứng bằng dữ liệu có đáp án biết trước. Ratio test theo ý tưởng [Lowe (2004), mục 7.1](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf) loại các lựa chọn gần như ngang nhau; paper dùng SIFT nên không coi ngưỡng của paper là bảo đảm chất lượng BRIEF. [OpenCV BFMatcher](https://docs.opencv.org/4.13.0/d3/da1/classcv_1_1BFMatcher.html) là nguồn đối chiếu Hamming và cross-check, không là thư viện được gọi trong code.

LSH có thể giảm tìm kiếm nhưng có nguy cơ bỏ lỡ best/second-best thật. Cross-check thêm điều kiện tương hỗ và có thể loại thêm cặp; phương pháp nhóm đã chọn để S5 xử lý outlier còn lại. Không đổi metric, thêm cross-check hoặc chuyển ANN riêng ở bản CUDA. Texture lặp, blur và ít đặc trưng vẫn có thể gây sai hoặc thiếu match; ratio không chứng minh correspondence đúng hình học.

Unit test S4 kiểm tra khoảng cách, ties, ratio và thứ tự output; log được in trong mục 3.4. Kiểm chứng hình học của pipeline nằm ở mục 3.5.

#### GPU Suitability

Các cặp descriptor độc lập, dùng XOR và `__popc` cho từng word trên GPU. Mapping V1 dự kiến một block/A, các thread chia B, reduction top-two rồi compact theo prefix để giữ thứ tự. V0 vẫn là scalar gốc. S4 là hot stage ở small/medium theo kết quả profiling ở mục 3.5, nên có cơ sở ưu tiên CUDA; thiết kế chi tiết, timing và rủi ro ở [mục 6 S4](#gpu-plan), dùng để tổng hợp mục 3.7. Chưa có speedup GPU đã đo.

### 3.3.5 S5 Homography

#### Method and Parameters

S5 dùng **uniform 4-point RANSAC, normalized DLT bằng double, MSAC scoring, adaptive stopping và all-inlier refit**. Input là keypoint A/B và Match của S4; output H biến điểm B sang hệ A. Bốn match chọn để fit có thể sai hoặc suy biến, nên mỗi H phải được chấm trên toàn bộ danh sách match.

| Thành phần | Giá trị/quy tắc |
|---|---|
| Seed | 16; SplitMix64 với stream theo `(seed,hypothesis_id)` |
| Sample | Bốn index khác nhau, uniform không hoàn lại trong mỗi sample |
| Cap | 2000 attempts, bao gồm mẫu bị loại |
| Solver | Homogeneous DLT, RMS normalization, one-sided Jacobi SVD tự cài, nội bộ double |
| Scorer | Tối thiểu $\sum_{j=0}^{m-1}\min(e_j^2,\tau^2)$; $e_j$ là forward reprojection error của match thứ $j$ theo chiều B → A |
| Tau | 2/3/6 px cho small/medium/large; scale từ 3 px tại $1920\times 1080$ theo đường chéo A |
| Confidence stopping | 0.99 |
| Refit | Fit toàn bộ inlier, reclassify một lần, refit lần hai nếu mask đổi |
| H bàn giao | `float[9]`, row-major, $\max_{i,j}\lvert h_{ij}\rvert=1$ và cố định dấu |
| Numerical epsilon | `homography_epsilon=1e-6f` trong cấu hình benchmark |

Inlier dùng bất đẳng thức chặt $e^2<\tau^2$; chiếu invalid bị tính loss $\tau^2$ và là outlier. Tau là ngưỡng RANSAC, khác ngưỡng nghiệm thu sai số H với ground truth. Duplicate/area/rank tolerances giữ theo implementation S5; không nới riêng ở GPU.

#### Fitting, Scoring and Stopping

Chuẩn hóa độc lập điểm A/B: trừ centroid và scale để RMS distance tới gốc bằng $\sqrt{2}$. Lập hai hàng DLT cho mỗi correspondence $B=(x,y)\to A=(u,v)$:

$$
L_j = \begin{bmatrix}
x & y & 1 & 0 & 0 & 0 & -ux & -uy & -u \\
0 & 0 & 0 & x & y & 1 & -vx & -vy & -v
\end{bmatrix}.
$$

SVD lấy right singular vector ứng với singular value nhỏ nhất; minimal fit $8\times 9$ cần đầy đủ right basis $9\times 9$. Giải theo dạng homogeneous nên không ép $h_{22}=1$. Sau đó denormalize $H=T_A^{-1}H_nT_B$ và kiểm tra hữu hạn, rank, tính khả nghịch cùng denominator. Normalization dựa trên [Harker và O'Leary (2005), mục 2.3](https://www.robots.ox.ac.uk/~phst/BMVC2005/papers/190/paper190.pdf); Jacobi SVD do nhóm tự cài theo cơ sở trong [Drmac và Veselic, LAWN 170](https://www.netlib.org/lapack/lawnspdf/lawn170.pdf), không link Eigen/LAPACK.

Trong mỗi attempt, mẫu trùng/gần trùng tọa độ, có bộ ba gần thẳng hàng hoặc solver lỗi bị loại. Mẫu hợp lệ tạo model và được chấm MSAC trên toàn bộ m match. Model tốt hơn có loss nhỏ hơn, rồi count lớn hơn, rồi hypothesis ID nhỏ hơn nếu bằng nhau. Candidate dưới bốn inlier không được chọn.

Budget được cập nhật theo best model hiện tại với $w=\operatorname{count}(\mathrm{best})/m$:

$$
K_{\mathrm{needed}} = \left\lceil \frac{\ln(1-c)}{\ln(1-w^4)} \right\rceil,
\qquad c=0.99,\quad 0<w<1.
$$

Code tính hai logarithm bằng `log1p(-c)` và `log1p(-(w*w*w*w))`; các trường hợp biên và giới hạn budget được xử lý riêng.

Implementation có guard cho $w=0$, $w=1$, non-finite và giới hạn $K_{\mathrm{needed}}\in[1,2000]$. Mỗi attempt được xử lý theo ID tăng dần rồi kiểm tra điều kiện dừng; budget có thể tăng khi best MSAC mới có count thấp hơn. Confidence này là giả định của stopping heuristic, không chứng minh xác suất H đạt chất lượng.

Các loss double được cộng theo cây cố định, leaves theo Match index và pad zero tới power-of-two. GPU cần giữ cây này cùng thứ tự commit hypothesis; không chọn model theo thread hoàn thành trước. Phép sampling, normalization, score, threshold và refit giữ cùng policy qua các version.

#### Refitting and Output

Lấy toàn bộ inlier $I_0$ của best model để fit $H_1$, reclassify toàn bộ match thành $I_1$. Nếu $I_1\ne I_0$ thì fit toàn bộ $I_1$ thêm một lần; nếu không đổi thì giữ $H_1$. Refit lỗi được trả về caller theo mã lỗi hiện có, không âm thầm dùng hypothesis cũ. Cuối cùng chuẩn hóa H, cast float32 và kiểm tra lại H thực sự bàn giao cho S6; đánh giá cuối không kích hoạt vòng refit thứ ba.

Chi phí scoring $O(Km)$ với K attempts thực sự xét; minimal fit có kích thước cố định, refit thêm chi phí DLT/SVD theo số inlier. Scratch V0 $O(m)$. DLT refit tối ưu sai số đại số nên không bảo đảm MSAC loss giảm sau mọi lần fit; log giữ score trước/sau để kiểm tra.

#### Rationale and Limitations

RANSAC tách các correspondence sai mà S4 chưa loại được. MSAC phân biệt các model có cùng số inlier qua mức residual, thay vì chỉ đếm inlier. Uniform sampler và solver nhỏ dễ tự cài, tạo fixture và đưa sang CUDA. [Brown và Lowe (2007), mục 3.1](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf) là nguồn về RANSAC/DLT cho stitching; [OpenCV USAC](https://docs.opencv.org/4.13.0/de/d3e/tutorial_usac.html) giúp đối chiếu các thành phần sampling/scoring/stopping, không phải implementation được gọi.

PROSAC cần thứ tự chất lượng cho sampler; MAGSAC++/VSAC thêm policy và cấu trúc verification/refinement. Nhóm giữ phương án hiện tại để kiểm soát CPU–GPU parity; chưa có thực nghiệm chứng minh nó tốt hơn các phương án kia. Ít inlier, điểm tập trung một vùng, parallax và vật chuyển động vẫn có thể làm H sai dù solver thành công. Sai số trên cặp synthetic đạt ngưỡng đã chốt được trình bày ở [mục correctness](#results), không dùng inlier count làm ground truth.

#### GPU Suitability

Hypotheses và residual có thể tính độc lập, còn model selection/adaptive stop cần commit theo prefix ID và refit cần solver double. V1 dự kiến block/hypothesis, thread chia match để chấm điểm và reduction theo cây cố định; phần solver/điều phối ban đầu dùng thread 0 trên Device. S5 chiếm tỷ trọng nhỏ trong benchmark V0, nên chưa ưu tiên tối ưu sâu trước profiling V1. Chi tiết ở [mục 6 S5](#gpu-plan) để tổng hợp mục 3.7; chưa có kết quả CPU–GPU parity hoặc speedup CUDA.

### 3.3.6 S6 Warp Blend

#### Method and Parameters

S6 nhận hai ảnh màu gốc A, B và homography H của S5 (đưa điểm của B sang hệ A), rồi tạo panorama trên một canvas chung. Phương pháp gồm bốn bước: tính canvas và offset, chép A vào canvas, inverse warping B với bilinear tự tính, và trộn trung bình 50/50 ở vùng chồng lấn.

| Thành phần | Giá trị |
|---|---|
| Hệ tọa độ | Tâm pixel ở tọa độ nguyên, gốc trên trái |
| Canvas | Hợp bốn góc của A và bốn góc của B sau khi chiếu qua H; $W_c = \lceil x_{\max} \rceil - \lfloor x_{\min} \rfloor + 1$, tương tự cho $H_c$; offset $(-\lfloor x_{\min} \rfloor, -\lfloor y_{\min} \rfloor)$ |
| Lấy mẫu B | Inverse mapping qua $H^{-1}$, bilinear trên 4 pixel lân cận, tính bằng `double` |
| Blend | Chỉ A hợp lệ: màu A; chỉ B hợp lệ: màu B; cả hai: $(C_A + C_B)/2$; không nguồn nào: đen |
| Làm tròn | $\lfloor \mathrm{clamp}(C, 0, 255) + 0.5 \rfloor$, chỉ một lần khi ghi output |
| Ngưỡng số học | `EPS_COORD` = 1e-6 pixel, `EPS_PIVOT` = 1e-12, `EPS_RCOND` = 1e-12 |
| Giới hạn canvas | Mỗi chiều ≤ 10 000 pixel, diện tích ≤ `max_canvas_pixels` = 100 triệu pixel |

Một pixel nguồn là hợp lệ khi tọa độ chiếu về nằm trong miền tâm pixel $[0, W-1] \times [0, H-1]$ của ảnh đó. Validity được xác định bằng hình học, không suy ra từ màu, nên pixel đen thật trong ảnh vẫn tham gia blend.

#### Algorithm and Cost

```text
normalize H, invert it with scaled partial pivoting, check the condition estimate
project the 4 corners of B; reject if the projective denominator changes sign
compute canvas size and offset; reject a canvas over the limits
allocate a black canvas and copy A at the offset
for each canvas pixel (xc, yc):
    pA = (xc - ox, yc - oy);  pB = H^-1 * pA
    if pB is valid in B:
        cB = bilinear(B, pB)
        pixel = (A covers pA) ? (cA + cB) / 2 : cB
```

Với canvas $P = W_c \times H_c$ pixel, chi phí là $O(P)$, bộ nhớ thêm chỉ là output $3P$ byte; không lưu ảnh B đã warp riêng. Chiều nghịch (đi từ pixel đích về nguồn) bảo đảm mỗi pixel canvas được tính đúng một lần, không có lỗ như khi chiếu từng pixel nguồn sang đích.

#### Rationale

Inverse warping cho mỗi pixel đích một giá trị xác định, nên không có lỗ hay pixel bị ghi đè nhiều lần như forward warping kiểu scatter. Bilinear mượt hơn nearest neighbour mà chỉ cần 4 mẫu, và nhóm tự tính để V0 và GPU dùng cùng công thức; bilinear phần cứng của texture chỉ có 8 bit phần lẻ cho trọng số nên không cho kết quả giống V0.

Các cách trộn đã xét gồm trung bình 50/50, feather theo khoảng cách tới biên ảnh, seam finding cùng multi-band và adaptive blending. Thiết kế ban đầu chọn feather; từ ngày 13/10 bản cài chuyển sang trung bình 50/50. Cách này tốn $O(1)$ mỗi pixel, không cần thêm trạng thái hay buffer, nên dễ giữ cùng một công thức từ V0 đến V3. Giới hạn của nó là không che được sai lệch: nếu H sai, cảnh có parallax hoặc vật chuyển động, vùng chồng lấn sẽ có ảnh kép, và chênh sáng giữa hai ảnh tạo đường chuyển tiếp. Seam finding và multi-band có thể giảm các lỗi này nhưng cần thêm solver, mask và nhiều tầng ảnh, khó giữ cùng một cài đặt từ V0 đến V3.

Kết quả thực tế của các cặp được trình bày trong mục 3.5.

#### GPU Suitability

Mỗi pixel canvas được tính độc lập và chỉ ghi một lần, nên V1 dùng một thread cho một pixel canvas, không cần atomic. Phần tính canvas và nghịch đảo H rất nhỏ, chạy trong một kernel một thread; Host chỉ đọc lại kích thước canvas để cấp phát output. Rủi ro chính là S6 dùng `double`, mà T4 có throughput phép tính 64 bit thấp hơn nhiều so với 32 bit. Chi phí S6 tăng theo diện tích canvas; timing ở mục 3.5 là căn cứ xác định mức ưu tiên. Mapping chi tiết và hướng tối ưu nằm ở [mục 6 tài liệu S6](#gpu-plan).

<a id="implementation"></a>
## 3.4 Sequential Implementation

### Program and Command Line

V0 là chương trình C++ đơn luồng chạy đủ 6 stage, build thành một file thực thi `stitch` từ thư mục `src/`. Notebook build trực tiếp từ các file trong `src/` đã nộp:

```bash
nvcc -arch=sm_75 -std=c++11 -O2 --fmad=false \
  -Xcompiler=-ffp-contract=off,-fno-tree-vectorize,-fno-tree-slp-vectorize \
  src/main.cu src/*.cpp -o build/stitch
```

Cờ `--fmad=false` và `-ffp-contract=off` tắt việc compiler tự gộp phép nhân và cộng thành FMA, để kết quả số thực của V0 làm chuẩn so sánh cho các phiên bản GPU. Hai cờ `-fno-tree-vectorize` và `-fno-tree-slp-vectorize` tắt tự động vector hóa, để V0 là baseline đơn luồng không dùng SIMD.

Cách gọi:

```text
stitch <left> <right> <output> --version 0 [--runs N] [--debug]
```

Chương trình đọc hai ảnh bằng `stb_image`, chạy một lần warm-up không đo, sau đó chạy `N` lần có đo (mặc định 5) và in mỗi lần một dòng CSV ra stdout: `Run_Index,S1_ms,S2_ms,S3_ms,S4_ms,S5_ms,S6_ms,Total_ms`, cuối cùng là hai dòng `Average` và `Minimum`. Panorama của lần chạy cuối được ghi ra file bằng `stb_image_write`. Cờ `--debug` in cấu hình, số keypoint, số match, thông tin RANSAC, H và kích thước canvas ra stderr sau khi các lần đo kết thúc; thông tin RANSAC lấy từ lần warm-up. Thông báo tiến độ giữa các lần chạy cũng nằm ngoài vùng đo.

### Source Layout

| Nhóm file | Nội dung |
|---|---|
| `common.h`, `versions.h`, `cuda_utils.cuh`, `stb_impl.cpp` | Kiểu dữ liệu chung (`Image`, `GrayImage`, `Keypoint`, `Descriptor`, `Match`, `Homography`, `Panorama`, `Config`, `TimingStats`, `ErrorCode`), chọn phiên bản, tiện ích CUDA cho các phiên bản sau, phần cài đặt của stb |
| `main.cu` | Đọc tham số dòng lệnh, đọc và ghi ảnh, warm-up, vòng đo, in CSV và báo lỗi |
| `v0_config.*`, `v0_pipeline.*` | Cấu hình V0 và hàm chạy lần lượt S1 đến S6 trên một vùng nhớ dùng lại giữa các lần chạy |
| `v0_preprocess.*`, `gaussian_kernel.*`, `gaussian_blur.*`, `image_layout.*` | S1, cùng phần blur và kiểm tra layout ảnh dùng chung với S2, S3 |
| `v0_detect.*`, `fast_corner.*`, `feature_border.h` | S2 |
| `v0_describe.*`, `brief_pattern.*` | S3 |
| `v0_match.*`, `feature_support.*` | S4 |
| `v0_homography.*`, `v0_homography_internal.h` | S5 |
| `v0_warp_blend.*` | S6 |

Mỗi stage nằm trong file riêng của người phụ trách và chỉ giao tiếp qua kiểu dữ liệu trong `common.h`. Nhờ vậy từng stage được kiểm thử độc lập trước khi ghép, và các phiên bản GPU sau này thay từng stage mà không đổi phần còn lại.

### Timing

Mỗi stage và toàn bộ 6 stage được đo bằng `std::chrono::high_resolution_clock`. Vùng đo không gồm đọc, ghi ảnh và log debug. Các buffer có kích thước biết trước (ảnh xám, mảng keypoint, descriptor, match) được cấp phát một lần trước vùng đo và dùng lại qua các lần chạy. Buffer tạm bên trong một stage và output của S6 (kích thước chỉ biết sau khi có H) được cấp phát trong stage nên tính vào thời gian của stage đó. S1, S2, S3 chạy cho cả hai ảnh và thời gian ghi là tổng của hai ảnh.

### Sequential Execution and Repeatability

V0 không dùng OpenMP, `std::thread`, `std::async`, `std::execution` hay intrinsics SIMD, và không build với `-fopenmp`. Tìm các từ khóa này trong `src/` (trừ hai header stb) không có kết quả nào là code dùng các cấu trúc trên. Phần ngẫu nhiên duy nhất là chọn mẫu của RANSAC ở S5, dùng seed cố định 16 và in seed trong log debug. Mọi phép cộng dồn có thứ tự cố định, nên chạy lại trên cùng input cho cùng output.

### External Code

Chỉ có hai header `stb_image.h` (v2.30) và `stb_image_write.h` (v1.16) của Sean Barrett lấy nguyên từ [github.com/nothings/stb](https://github.com/nothings/stb), dùng để đọc và ghi ảnh; `stb_impl.cpp` chỉ bật phần cài đặt của hai header này. Toàn bộ code thuật toán do Nhóm tự viết. Các file cài theo mô tả thuật toán trong bài báo hoặc tài liệu (FAST, BRIEF, ratio test, normalized DLT, Jacobi SVD, SplitMix64, MSAC) ghi nguồn ở dòng `Adapted from:` đầu file và trong mục tài liệu tham khảo.

### Error Handling

Mỗi stage trả về một `ErrorCode` thay vì gọi `exit()`. `main.cu` đổi mã lỗi thành thông báo và exit code khác 0 cho các trường hợp: thiếu file, ảnh không đọc được, tham số dòng lệnh sai, phiên bản không tồn tại, `--runs` không dương, không đủ keypoint hoặc match, homography suy biến, canvas vượt giới hạn và cấp phát thất bại. Khi một stage lỗi, chương trình không ghi panorama.

### Tests

Mỗi stage có bộ test riêng trong `tests/` với đáp án tính độc lập, cộng thêm test tích hợp chạy S1 đến S6 trên cặp ảnh tổng hợp có phép dịch biết trước. Kết quả các test và sai số homography trên cặp synthetic nằm ở phần correctness của mục 3.5.

### Build the Submitted Source

Cell dưới build đúng `src/` của bài nộp. Kết quả test được in ở các cell kế tiếp.

In [ ]:
CPP_SOURCES = sorted(str(path.relative_to(PROJECT)) for path in (PROJECT / "src").glob("*.cpp"))
NVCC_FLAGS = ["-arch=sm_75", "-std=c++11", "-O2", "--fmad=false",
              "-Xcompiler=-ffp-contract=off,-fno-tree-vectorize,-fno-tree-slp-vectorize"]
STITCH = BUILD / "stitch"
build_command = ["nvcc", *NVCC_FLAGS, "src/main.cu", *CPP_SOURCES, "-o", STITCH]
result = execute(build_command, cwd=PROJECT)
(RESULTS / "build.log").write_text(result.stdout + result.stderr, encoding="utf-8")
(RESULTS / "build_command.json").write_text(json.dumps([str(x) for x in build_command], indent=2))
CHECKS["build"] = True

### 3.4.2 Unit and Integration Tests

Chạy bộ test đã có cho S1–S6, cấu hình và pipeline. Bộ test C++ được link với module thật; không tạo ground truth bằng chính hàm đang kiểm thử. Các test Python kiểm tra công cụ dataset/synthetic và helper notebook. Test không thuộc benchmark; lỗi test dừng phiên chạy.

In [ ]:
test_logs = []
result = execute([sys.executable, "tests/run_v0_features.py", "--compiler", "nvcc", "--require-t4"], cwd=PROJECT)
test_logs.append(result.stdout + result.stderr)
# The existing runner covers S4, S5, allocation, and S6. Cover the remaining modules here.
EXTRA_TESTS = ["test_v0_preprocess", "test_v0_detect", "test_v0_describe",
               "test_v0_detect_describe", "test_v0_config", "test_v0_pipeline"]
for name in EXTRA_TESTS:
    binary = BUILD / name
    execute(["nvcc", *NVCC_FLAGS, "-Isrc", *CPP_SOURCES, f"tests/{name}.cpp", "-o", binary], cwd=PROJECT)
    result = execute([binary], cwd=PROJECT)
    if "PASS" not in result.stdout:
        raise RuntimeError("Test did not report PASS: " + name)
    test_logs.append(result.stdout + result.stderr)
for test_file in ("test_dataset_scripts.py", "test_synthetic_scripts.py", "test_midterm_notebook.py",
                  "test_run_benchmark.py", "test_plot_benchmark.py"):
    result = execute([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", test_file], cwd=PROJECT)
    test_logs.append(result.stdout + result.stderr)
(RESULTS / "tests.log").write_text("\n".join(test_logs), encoding="utf-8")
CHECKS["tests"] = True

### 3.4.3 Command-Line and Error Handling

CLI: `stitch <left> <right> <output> --version 0 [--runs N] [--debug]`. Mặc định N = 5. Log debug đi vào stderr, CSV đi vào stdout. Các ca dưới kiểm tra cờ, version, số lần chạy và file lỗi; không đo các ca thất bại.

In [ ]:
left = DATASET / "site1/small_left.jpg"
right = DATASET / "site1/small_right.jpg"
output = RESULTS / "cli_output.png"
bad_file = RESULTS / "not_an_image.jpg"
bad_file.write_text("invalid image bytes", encoding="utf-8")
missing_file = RESULTS / "input_that_does_not_exist.jpg"
cases = [
    ("missing arguments", [], "InvalidArgument"),
    ("unknown flag", [left, right, output, "--version", "0", "--unknown"], "UnknownOption"),
    ("unsupported version", [left, right, output, "--version", "1"], "UnsupportedVersion"),
    ("non-positive runs", [left, right, output, "--version", "0", "--runs", "0"], "InvalidArgument"),
    ("missing input", [missing_file, right, output, "--version", "0"], "InputNotFound"),
    ("unreadable image", [bad_file, right, output, "--version", "0"], "ImageReadFailed"),
]
cli_rows = []
for label, arguments, expected in cases:
    result = execute([STITCH, *arguments], cwd=PROJECT, check=False)
    if result.returncode == 0 or f"[{expected}]" not in result.stderr:
        raise AssertionError("Unexpected CLI behavior: " + label)
    cli_rows.append({"case": label, "expected_error": expected, "exit_code": result.returncode})
display(pd.DataFrame(cli_rows))
CHECKS["cli_errors"] = True

<a id="results"></a>
## 3.5 Results and Profiling

### 3.5.1 Synthetic Homography Accuracy

Correctness có ba lớp: test từng stage với đáp án độc lập, test tích hợp từ ảnh tới pixel panorama, và sai số H so với ground truth trên cặp synthetic. Output test ở mục 3.4 thuộc phiên hiện tại. Inlier count chỉ mô tả estimator, không phải nhãn ground truth.



Nguồn C là ảnh `raw/tuan/standard1_left.jpg`, kích thước $8160\times 6144$, SHA-256 `e9fbe7ecfed6344a36610fe50f2c6019bcd1525f4e217c033ed8ded4f90bf1e8`. Generator cắt/thu nhỏ cảnh bằng Lanczos, lấy A là một crop và tự lấy mẫu B bằng bilinear theo $H_{\mathrm{gt}}$ B → A. $H_{\mathrm{gt}}$ được định nghĩa trước khi chạy estimator, không lấy từ solver S5 hay S6 đang kiểm thử. Ảnh sinh lưu PNG để tránh thêm nhiễu JPEG.

Hai loại hình học được sinh ở small $1280\times 720$, medium $1920\times 1080$ và large $3840\times 2160$:

| Case | Hình học ground truth | Điểm Q trong overlap |
|---|---|---:|
| translation | Dịch ngang $\operatorname{round}(0.55(W-1))$, lần lượt 703/1055/2111 px | 180/400 mỗi kích thước |
| projective | Trong tọa độ chia $W-1$: xoay 3° quanh tâm B, thành phần chiếu (0.10,0.05), dịch (0.55,0.02) | 165/400 mỗi kích thước |

Case projective kiểm tra thành phần phối cảnh ngoài phép dịch. Pipeline thật chạy S1–S6 từ hai ảnh; không đưa correspondence ground truth trực tiếp vào S5 rồi gọi đó là đánh giá end-to-end.

### Error Metric and Acceptance Threshold

Tạo lưới $20\times 20$ trên B, chỉ giữ Q gồm các điểm mà $H_{\mathrm{gt}}$ chiếu hợp lệ vào A. Tập Q được cố định theo ground truth, không chọn theo inlier mask hoặc $H_{\mathrm{est}}$. Đánh giá $H_{\mathrm{est}}$ float32 thực sự bàn giao qua log `[debug] H(B->A)=`:

$$
\begin{aligned}
e(q) &= \left\lVert \pi(H_{\mathrm{est}},q)-\pi(H_{\mathrm{gt}},q) \right\rVert_2, \\
\mathrm{RMSE}_{\mathrm{px}} &= \sqrt{\frac{1}{\lvert Q\rvert}\sum_{q\in Q} e(q)^2}, \\
\mathrm{Max}_{\mathrm{px}} &= \max_{q\in Q} e(q), \\
\mathrm{NRMSE} &= \frac{\mathrm{RMSE}_{\mathrm{px}}}{\sqrt{W_A^2+H_A^2}}.
\end{aligned}
$$

Trong đó $\pi(H,q)$ là phép chiếu điểm $q$ bằng homography $H$, còn $W_A,H_A$ là chiều rộng và chiều cao ảnh A.

Sai số có đơn vị pixel của A; NRMSE là sai số tương đối theo đường chéo ảnh. Không dùng hiệu chín hệ số H vì H xác định tới hệ số scale. Q rỗng hoặc bất kỳ phép chiếu $H_{\mathrm{est}}$ nào trong Q invalid làm phép đánh giá thất bại; không bỏ điểm lỗi để làm giảm sai số.

Theo P1-T01, quyết định đã chốt của nhóm, **mỗi cặp** phải có $\mathrm{RMSE}\le0.5\,\mathrm{px}$ và $\mathrm{Max}\le1\,\mathrm{px}$. Ngưỡng này khác tau 2/3/6 px dùng phân loại inlier trong RANSAC.



Cell dưới tự tạo các cặp, chạy pipeline từ ảnh rồi in H_gt, H_est và metric. RMSE/Max đo hình học trong overlap; không phải sai số màu panorama hoặc bảo đảm chất lượng trên mọi cảnh thật.

In [ ]:
from homography_common import load_truth
from homography_error import parse_estimate, evaluate

SYNTHETIC = RESULTS / "synthetic"
execute([sys.executable, "scripts/make_synthetic_pair.py", "--source", RAW_SYNTHETIC,
         "--out", SYNTHETIC], cwd=PROJECT)
synthetic_rows = []
for case in ("translation", "projective"):
    for size in SIZES:
        prefix = SYNTHETIC / case / size
        result = execute([STITCH, str(prefix) + "_left.png", str(prefix) + "_right.png",
                          str(prefix) + "_pano.png", "--version", "0", "--runs", "1", "--debug"], cwd=PROJECT)
        Path(str(prefix) + ".csv").write_text(result.stdout, encoding="utf-8")
        Path(str(prefix) + ".log").write_text(result.stderr, encoding="utf-8")
        parse_stitch_csv(result.stdout, runs=1)
        truth = load_truth(str(prefix) + "_truth.json")
        estimate = parse_estimate(result.stderr)
        metrics = evaluate(truth, estimate)
        passed = metrics["rmse_px"] <= 0.5 and metrics["max_px"] <= 1.0
        synthetic_rows.append({"case": case, "resolution": size, **metrics, "passed": passed})
        print("H_gt:", truth["H_gt"])
        print("H_est:", estimate.tolist())
synthetic_table = pd.DataFrame(synthetic_rows)
synthetic_table.to_csv(RESULTS / "synthetic_accuracy.csv", index=False)
display(synthetic_table)
CHECKS["synthetic_accuracy"] = len(synthetic_rows) == 6 and all(row["passed"] for row in synthetic_rows)
if not CHECKS["synthetic_accuracy"]:
    raise AssertionError("Synthetic accuracy failed; do not proceed with an accepted V0 baseline.")

In [ ]:
# Same binary, input and configuration; timing is intentionally not compared.
prefix = SYNTHETIC / "projective" / "small"
repeat_output = SYNTHETIC / "projective" / "small_repeat.png"
repeat_result = execute([STITCH, str(prefix) + "_left.png", str(prefix) + "_right.png",
                         repeat_output, "--version", "0", "--runs", "1", "--debug"], cwd=PROJECT)
original_log = Path(str(prefix) + ".log").read_text()
original_h = parse_estimate(original_log)
repeat_h = parse_estimate(repeat_result.stderr)
with PILImage.open(str(prefix) + "_pano.png") as original, PILImage.open(repeat_output) as repeated:
    same_pixels = original.size == repeated.size and np.array_equal(np.asarray(original), np.asarray(repeated))
match_pattern = r"\[debug\] S4 matches=(\d+)"
original_matches = int(re.search(match_pattern, original_log).group(1))
repeat_matches = int(re.search(match_pattern, repeat_result.stderr).group(1))
repeatability = {"same_h": bool(np.array_equal(original_h, repeat_h)),
                 "same_matches": original_matches == repeat_matches, "same_panorama_pixels": bool(same_pixels)}
display(pd.DataFrame([repeatability]))
(RESULTS / "repeatability.json").write_text(json.dumps(repeatability, indent=2))
CHECKS["repeatability"] = all(repeatability.values())
if not CHECKS["repeatability"]:
    raise AssertionError("Algorithm output changed with the same V0 input/configuration")

### 3.5.2 Site Panoramas and Challenging Pair

Sinh panorama của các cặp chuẩn và cặp khó với cùng cấu hình V0. Timing của lượt tạo ảnh minh họa không được dùng thay benchmark chính thức. Ảnh thu nhỏ dùng để xem tổng thể; crop giữ độ phân giải gốc để kiểm tra cạnh, ảnh kép và chuyển sáng.

In [ ]:
if PANORAMA_SIZE not in SIZES:
    raise ValueError("Unknown panorama resolution")
PANORAMAS = RESULTS / "panoramas"
PANORAMAS.mkdir()
panorama_rows = []
for site in SITES:
    prefix = PANORAMAS / f"{site}_{PANORAMA_SIZE}"
    result = execute([STITCH, DATASET / site / f"{PANORAMA_SIZE}_left.jpg",
                      DATASET / site / f"{PANORAMA_SIZE}_right.jpg", str(prefix) + ".png",
                      "--version", "0", "--runs", "1", "--debug"], cwd=PROJECT)
    Path(str(prefix) + ".csv").write_text(result.stdout, encoding="utf-8")
    Path(str(prefix) + ".log").write_text(result.stderr, encoding="utf-8")
    parse_stitch_csv(result.stdout, runs=1)
    with PILImage.open(str(prefix) + ".png") as image:
        dimensions = image.size
    panorama_rows.append({"site": site, "resolution": PANORAMA_SIZE,
                          "canvas_width": dimensions[0], "canvas_height": dimensions[1]})
    display(Markdown(f"**{site} · {PANORAMA_SIZE} · canvas {dimensions[0]} × {dimensions[1]}**"))
    display(DisplayImage(data=preview_image(str(prefix) + ".png")))
display(pd.DataFrame(panorama_rows))
CHECKS["panoramas"] = len(panorama_rows) == 4

### 3.5.3 Native-Resolution Inspection Crops

Cell dưới lấy các crop 512×512 gần phía trên, giữa và dưới bao giao hình học của hai ảnh, hiển thị **không resize**. Bao giao trục tọa độ không phải validity mask chính xác, nên cần kiểm tra vùng ảnh thật trong mỗi crop. Có thể thêm crop thủ công ở vị trí người/xe; không suy luận chất lượng từ thumbnail toàn cảnh.

In [ ]:
# Override a site with canvas-coordinate boxes (x0, y0, x1, y1) after the first inspection.
MANUAL_CROPS = {}  # Example: {"challenge": [(1200, 500, 1712, 1012)]}
crop_records = []
for site in SITES:
    prefix = PANORAMAS / f"{site}_{PANORAMA_SIZE}"
    width, height = SIZES[PANORAMA_SIZE]
    envelope = overlap_box(Path(str(prefix) + ".log").read_text(), width, height, width, height)
    x0, y0, x1, y1 = envelope
    boxes = MANUAL_CROPS.get(site, [])
    if not boxes:
        for fraction in (0.2, 0.5, 0.8):
            crop_w, crop_h = min(512, x1 - x0), min(512, y1 - y0)
            cx = max(x0, min((x0 + x1) // 2 - crop_w // 2, x1 - crop_w))
            cy = max(y0, min(int(y0 + fraction * (y1 - y0)) - crop_h // 2, y1 - crop_h))
            boxes.append((cx, cy, cx + crop_w, cy + crop_h))
    with PILImage.open(str(prefix) + ".png") as image:
        for index, box in enumerate(boxes, 1):
            a, b, c, d = box
            if not (0 <= a < c <= image.width and 0 <= b < d <= image.height):
                raise ValueError("Inspection crop leaves the canvas")
            crop_path = PANORAMAS / f"{site}_crop_{index}.png"
            image.crop(box).save(crop_path)
            display(Markdown(f"**{site} · crop {index} · {box} · 1:1 pixels**"))
            display(DisplayImage(filename=str(crop_path)))
            crop_records.append({"site": site, "box": list(box), "file": crop_path.name})
(RESULTS / "inspection_crops.json").write_text(json.dumps(crop_records, indent=2))
CHECKS["inspection_crops"] = len(crop_records) >= 4
CHECKS["quality_review"] = QUALITY_REVIEW_CONFIRMED

### Panorama Quality Analysis

Nhận xét được đối chiếu với lần Run all trên Colab T4 ngày 24/10, phiên `20261024T113622Z-5c29204a`: ảnh panorama medium của bốn cặp và ba crop 512×512 cho mỗi cặp, hiển thị không resize. Crop chỉ là vùng kiểm tra được chọn, không thay cho đánh giá mọi pixel.

#### Execution Results

| Cặp | Địa điểm | Match S4 | Inlier S5 | Canvas |
|---|---|---:|---:|---|
| site1 | Nhà thờ, ngoài trời | 635 | 274 | 3692×1496 |
| site2 | Ẩn Coffee, trong nhà | 383 | 153 | 4162×2148 |
| site3 | Tiệm cà phê Mai An, trong nhà | 471 | 160 | 3558×1791 |
| challenge | Khu ký túc xá, ban đêm | 614 | 391 | 2746×1489 |

Cả 4 cặp đều chạy hết 6 stage và tạo panorama. Ở cả 4 cặp, ảnh B nằm bên phải A và bị chiếu lệch lên trên một phần, nên canvas có offset y dương và có các vùng đen không thuộc ảnh nào. Vùng đen này là phần canvas ngoài hai ảnh, không phải lỗi.

#### Pair Observations

##### site1

Panorama liền khung và phần kiến trúc ở xa nhìn tương đối khớp trên ảnh tổng thể. Tuy nhiên crop độ phân giải gốc cho thấy tượng, dây điện và một số cạnh hàng rào/ghế có viền nhòe hoặc kép; không thể giữ nhận xét cũ rằng mọi crop sắc nét và không có ảnh kép. Ảnh thu nhỏ che bớt sai lệch, nên chất lượng phải đọc cùng crop gốc. Chỉ từ các ảnh này chưa đủ tách hoàn toàn blur lúc chụp khỏi sai lệch do warp/blend.

##### site2

Đây là cặp có lỗi rõ nhất. Trong vùng chồng lấn, khung cửa sổ và bức tường phía xa khớp nhau, nhưng các vật ở gần máy bị nhân đôi: đế đèn trên trần, các đèn thả, hàng ly trên quầy và cạnh quầy đều hiện hai bản lệch nhau. Đây là lỗi parallax: cảnh trong nhà có nhiều mặt phẳng ở các khoảng cách khác nhau, trong khi một homography chỉ khớp được một mặt phẳng (ở đây là tường và cửa sổ). Khi tâm máy dịch đi một chút lúc xoay người chụp, vật ở gần dịch nhiều hơn vật ở xa. Trung bình 50/50 giữ cả hai bản nên ảnh kép hiện rõ. Ngoài ra có một đường chuyển sáng ở mép trên của ảnh A trên trần nhà, do hai ảnh có độ sáng khác nhau.

##### site3

Tường, trần, đèn chùm và khung tranh khớp tốt. Ghế và bàn ở tiền cảnh có viền mờ nhẹ do cùng lỗi parallax như site2 nhưng mức nhẹ hơn. Ở mép phải của ảnh A trên tường có một đường chuyển sáng thẳng đứng, do hai ảnh có độ sáng khác nhau.

##### Challenging Pair

Cặp chụp ban đêm, chênh sáng lớn giữa bầu trời, tòa nhà có đèn và tán cây, có người đi bộ và xe máy di chuyển. Pipeline vẫn tìm được nhiều inlier nhất trong 4 cặp (391 trên 614 match) và panorama khớp hình học: tòa nhà, cột đèn, thùng rác và ghế đá đều liền mạch. Người đi bộ và xe máy nằm ngoài vùng chồng lấn nên không bị nhân đôi; xe máy bị nhòe là do chuyển động lúc chụp. Các hạn chế quan sát được: nhiễu ảnh đêm trên tán cây tối, một đường chuyển sáng trên bầu trời ở mép trên của ảnh A, và chân cột đèn có viền mờ nhẹ.

#### Overall Assessment

V0 tạo được panorama cho cả bốn cặp, nhưng chất lượng không đồng đều. Site1 còn cạnh nhòe/kép trong crop, site3 có viền mờ ở tiền cảnh, site2 có ảnh kép rõ ở các vật gần máy; challenge giữ được bố cục cảnh dù còn giới hạn ảnh đêm và chuyển sáng. Việc tường và cửa sổ phía xa khớp trong khi vật gần máy bị lệch phù hợp với ảnh hưởng của parallax và giới hạn của một homography toàn cục. Bộ synthetic đạt ngưỡng cho thấy pipeline ước lượng đúng hình học trên các cặp được kiểm soát; kết quả đó không tự chứng minh H trên cảnh thật là chính xác tuyệt đối hoặc loại trừ mọi lỗi ước lượng.

Hai giới hạn của phương pháp hiện tại hiện rõ trong kết quả:

- Một homography toàn cục không mô tả được cảnh có nhiều độ sâu khi tâm máy dịch chuyển. Giảm lỗi này cần chụp lại với tâm máy cố định hơn, hoặc dùng mô hình căn chỉnh cục bộ, nằm ngoài phạm vi pipeline hiện tại.
- Trung bình 50/50 không che ảnh kép và không làm mờ chênh sáng ở mép ảnh. Feather theo khoảng cách hoặc seam finding sẽ giảm đường chuyển sáng, nhưng đổi blend là đổi phương pháp, phải cả Nhóm quyết định và áp dụng từ V0.

Các phiên bản GPU giữ nguyên phương pháp nên sẽ có cùng các lỗi này; so sánh V1 với V0 dựa trên việc output giống nhau, không phải trên chất lượng tốt hơn.

### 3.5.4 Official Timing Protocol

Benchmark chính: **site1/site2/site3 × small/medium/large**, một warm-up không đo và `--runs 5`. Đo bằng `std::chrono::high_resolution_clock` trong C++; không lấy thời gian Python/subprocess làm thời gian thuật toán. Đọc/ghi ảnh và debug ngoài vùng đo. Toàn bộ bảng phải đến từ cùng một phiên Colab T4, cùng nguồn và cờ build.

Script `scripts/run_benchmark.py` (P3-T04) chạy `stitch` cho 9 cấu hình và lưu nguyên stdout CSV của mỗi cấu hình vào `<out>/<site>/<size>.csv`, kèm `summary.csv`, `summary.md` và `environment.json` (tên GPU, CPU và các lệnh đã chạy). Notebook đọc lại các CSV này, kiểm tra đủ 9 cấu hình và đúng số lần đo rồi dựng bảng ở mục 3.5.5. Không dùng CSV từ phiên cũ; khi đang đo, không chạy cell nặng nào khác.

In [ ]:
BENCHMARK_ROOT = RESULTS / "benchmark"
BENCHMARK_ROOT.mkdir()
benchmark_script = PROJECT / "scripts/run_benchmark.py"
BENCHMARK_EXECUTED = False
benchmark_average = benchmark_minimum = benchmark_runs = None
if not benchmark_script.is_file() or BENCHMARK_ARGS is None:
    print("Chưa chạy benchmark: thiếu runner hoặc danh sách đối số; không có timing thay thế.")
    print("Binary:", STITCH, "Dataset:", DATASET, "Runs:", RUNS, "Output:", BENCHMARK_ROOT)
else:
    if not isinstance(BENCHMARK_ARGS, list) or not all(isinstance(x, str) for x in BENCHMARK_ARGS):
        raise TypeError("BENCHMARK_ARGS must be the runner argument list.")
    # Substitutions are explicit argument values, never a shell command.
    substitutions = {"stitch": str(STITCH), "dataset": str(DATASET),
                     "runs": str(RUNS), "out": str(BENCHMARK_ROOT)}
    arguments = [argument.format(**substitutions) for argument in BENCHMARK_ARGS]
    result = execute([sys.executable, benchmark_script, *arguments], cwd=PROJECT, timeout=7200)
    (RESULTS / "benchmark_runner.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    average, minimum, measured = collect_benchmark_csv(BENCHMARK_ROOT, RUNS)
    benchmark_average, benchmark_minimum, benchmark_runs = map(pd.DataFrame, (average, minimum, measured))
    benchmark_average.to_csv(RESULTS / "timing_average.csv", index=False)
    benchmark_minimum.to_csv(RESULTS / "timing_minimum.csv", index=False)
    benchmark_runs.to_csv(RESULTS / "timing_runs.csv", index=False)
    BENCHMARK_EXECUTED = True
CHECKS["official_benchmark"] = BENCHMARK_EXECUTED

### 3.5.5 Per-Stage Timing and Hot Stages

Các bảng và biểu đồ bên dưới lấy từ CSV của chính phiên này. Thời gian từng stage tính cho cả cặp ảnh; Total đo riêng quanh pipeline, không ép bằng tổng các stage. Bảng tỷ trọng và hot stage giúp kiểm tra nhận xét khi timing dao động giữa các lần chạy.

In [ ]:
if BENCHMARK_EXECUTED:
    display(Markdown("**Average · milliseconds**"))
    display(benchmark_average.round(4))
    display(Markdown("**Minimum · milliseconds**"))
    display(benchmark_minimum.round(4))
    stage_columns = TIMING_COLUMNS[:-1]
    hot = benchmark_average[["site", "resolution"]].copy()
    hot["hot_stage"] = benchmark_average[stage_columns].idxmax(axis=1)
    hot["hot_stage_fraction"] = benchmark_average[stage_columns].max(axis=1) / benchmark_average["Total_ms"]
    display(hot)
    stage_percentages = benchmark_average[["site", "resolution"]].copy()
    for column in stage_columns:
        stage_percentages[column] = 100.0 * benchmark_average[column] / benchmark_average["Total_ms"]
    display(Markdown("**Stage fractions · percent of Total**"))
    display(stage_percentages.round(3))
    by_resolution = stage_percentages.groupby("resolution")[stage_columns].mean().loc[list(SIZES)]
    display(Markdown("**Mean stage fractions across sites · percent**"))
    display(by_resolution.round(3))
    for size in SIZES:
        selected = hot[hot.resolution == size]
        observation = "; ".join(f"{row.site}: {row.hot_stage}" for row in selected.itertuples())
        display(Markdown(f"Hot stage ở {size} trong phiên này: {observation}."))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    colors = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]
    for axis, site in zip(axes, SITES[:3]):
        table = benchmark_average[benchmark_average.site == site].set_index("resolution").loc[list(SIZES)]
        table[stage_columns].plot.bar(stacked=True, ax=axis, color=colors, rot=0, legend=False)
        axis.set(title=site, xlabel="Resolution", ylabel="Average time (ms)")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.savefig(RESULTS / "stage_timing.png", dpi=180, bbox_inches="tight")
    plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for axis, site in zip(axes, SITES[:3]):
        table = stage_percentages[stage_percentages.site == site].set_index("resolution").loc[list(SIZES)]
        table[stage_columns].plot.bar(stacked=True, ax=axis, color=colors, rot=0, legend=False)
        axis.set(title=site, xlabel="Resolution", ylabel="Percent of Total")
    fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.savefig(RESULTS / "stage_fractions.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    display(Markdown("Chưa có bảng timing/hot stage: benchmark của phiên này chưa hoàn tất."))

### Timing Interpretation

S4 là hot stage ở small và medium của cả ba site. Tỷ trọng trung bình S4 giảm từ 55.7% ở small xuống 14.3% ở large. S4 xét mọi cặp descriptor nên phụ thuộc số keypoint, không phụ thuộc trực tiếp số pixel. Site1 và site2 đều đạt giới hạn keypoint ở các kích thước, nên S4 nằm trong khoảng 701.222–739.281 ms. Site3 có ít descriptor hơn nên S4 nhanh hơn.

Ở large, S2 chiếm nhiều thời gian nhất tại site1; S6 đứng đầu tại site2 và site3. Các lượt xử lý ảnh của S1–S3 tăng theo số pixel, còn S6 phụ thuộc diện tích canvas. Do đó tăng độ phân giải làm chi phí phía pixel rõ hơn so với S4. S5 chiếm 1.2–7.1% Total trong phiên này; không nên ưu tiên tối ưu sâu trước các stage đang chiếm phần lớn thời gian.

### 3.5.6 Amdahl's Law

Nhóm ưu tiên tối ưu S2, S4 và S6 dựa trên profiling. Với tập này, $p=(T_{S2}+T_{S4}+T_{S6})/T_{total}$ và giới hạn lý tưởng là $1/(1-p)$ nếu phần được chọn nhanh vô hạn còn phần khác giữ nguyên. Cell dưới tính từng site và kích thước từ bảng timing hiện tại.

Không lấy tổng mọi stage làm p: tổng đó gần bằng Total của V0 nên không phản ánh được giới hạn có ý nghĩa. Với pipeline GPU, truyền dữ liệu và overhead launch phải được đo riêng. Đây là ước lượng, chưa phải speedup CUDA đã thực nghiệm.

In [ ]:
if BENCHMARK_EXECUTED and ACCELERATED_STAGES:
    if (len(set(ACCELERATED_STAGES)) != len(ACCELERATED_STAGES) or
            any(type(stage) is not int or stage not in range(1, 7) for stage in ACCELERATED_STAGES)):
        raise ValueError("Accelerated stages must be distinct integers from 1 to 6")
    selected_columns = [f"S{stage}_ms" for stage in ACCELERATED_STAGES]
    amdahl = benchmark_average[["site", "resolution"]].copy()
    if (benchmark_average.Total_ms <= 0).any():
        raise ValueError("Amdahl requires positive end-to-end measurements")
    amdahl["p"] = benchmark_average[selected_columns].sum(axis=1) / benchmark_average.Total_ms
    if ((amdahl.p < 0) | (amdahl.p > 1 + 1e-6)).any():
        raise ValueError("Stage fractions are inconsistent with total timing")
    amdahl["p"] = amdahl.p.clip(upper=1)
    amdahl["ideal_speedup_limit"] = amdahl.p.map(lambda p: float("inf") if p == 1 else 1 / (1 - p))
    display(amdahl)
    amdahl.to_csv(RESULTS / "amdahl.csv", index=False)
    CHECKS["amdahl_decision"] = True
else:
    print("Chưa có benchmark của phiên này; chưa tính Amdahl.")
    CHECKS["amdahl_decision"] = False

### Amdahl Interpretation

Giới hạn lý tưởng của tập stage này nằm trong khoảng 5.95–10.42 lần ở small và 4.99–6.36 lần ở large. Khi ảnh lớn, phần thời gian ngoài tập được chọn tăng, nên giới hạn giảm. Đây là ước lượng nếu phần được chọn nhanh vô hạn và phần còn lại giữ nguyên, chưa phải speedup CUDA đã đo.

Không tính p cho toàn bộ stage vì tổng thời gian stage gần bằng Total của V0. Với V1 cần đo thêm truyền Host–Device, overhead launch và phần tuần tự còn lại; không suy giới hạn thực tế từ biểu thức gần chia cho không của V0.

<a id="dataset"></a>
## 3.6 Dataset

### Image Collection

Mỗi thành viên tự chụp ảnh ở địa điểm của mình theo cùng một quy định: đứng yên một chỗ và xoay máy giữa hai lần chụp, chồng lấn khoảng 30–50%, dùng camera chính 1x, không zoom, chụp hai ảnh liền nhau. Ảnh gốc được nộp vào thư mục chung trên Google Drive, đặt tên theo dạng `<loại cặp><số>_<left|right>.jpg`.

Có 9 địa điểm, 13 cặp chính (9 cặp chuẩn và 4 cặp khó) cùng một số cặp dự phòng:

| Người chụp | Địa điểm | Cặp chính |
|---|---|---|
| Tuấn | Ẩn Coffee (quán cà phê, trong nhà và ngoài trời) | 2 chuẩn, 1 khó |
| Thành | Nhà thờ; đường ngoài trời; quán cafe Houjicha (trong nhà); sau khuôn viên Trường Đại học Đồng Nai; trên đường đi | 4 chuẩn, 1 khó |
| Chương | Khuôn viên Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM; khu vực ký túc xá | 1 chuẩn, 1 khó |
| Đế | Tiệm cà phê Mai An (trong nhà và ngoài trời) | 2 chuẩn, 1 khó |

### Pair Selection

Script `scripts/check_pairs.py` kiểm tra từng cặp chính theo các yêu cầu có thể kiểm tra tự động: tên file, định dạng JPEG, ảnh ngang, hai ảnh cùng kích thước và cùng thiết bị, chụp liền nhau, độ nét tương đối và ước lượng độ chồng lấn. Những tiêu chí script không tự kiểm tra được (cảnh tĩnh, ít parallax, đủ texture, không lộ mặt người lạ) được đánh dấu để thành viên xem lại bằng mắt. Script xếp thứ tự đề xuất, sau đó Nhóm chạy thử pipeline V0 trên cả 13 cặp ở 3 kích thước để xem cặp nào ra panorama.

Ngày 18/10 cả Nhóm chốt bộ dataset sau:

| Vai trò | Cặp | Địa điểm | Lý do chọn |
|---|---|---|---|
| Site 1, chuẩn | `thanh-standard1` | Nhà thờ, ngoài trời | Xếp đầu theo script, V0 ra panorama ở cả 3 kích thước |
| Site 2, chuẩn | `tuan-standard2` | Ẩn Coffee, trong nhà | V0 ra panorama ở cả 3 kích thước |
| Site 3, chuẩn | `de-standard2` | Tiệm cà phê Mai An, trong nhà | V0 ra panorama ở cả 3 kích thước |
| Cặp khó | `chuong-challenge1` | Khu vực ký túc xá, ngoài trời | Cảnh đêm, chênh sáng mạnh, có người và xe chuyển động |

Ba cặp chuẩn ở ba địa điểm khác nhau, đúng yêu cầu của đề. Một số cặp không được chọn vì pipeline thất bại: `tuan-standard1` có mái pergola sát máy gây parallax, `thanh-standard3` làm phép chiếu đưa một phần ảnh phải qua đường chân trời, còn `tuan-challenge1` có quá ít texture.

### Dataset Preparation

Script `scripts/prepare_dataset.py` xử lý mỗi cặp được chọn như sau:

1. Xoay ảnh theo thông tin EXIF orientation, vì stb đọc ảnh mà bỏ qua EXIF.
2. Cắt cả hai ảnh về tỷ lệ 16:9 bằng cùng một khung cắt ở giữa ảnh, không phóng to.
3. Thu nhỏ bằng Lanczos ra ba kích thước 1280×720 (small), 1920×1080 (medium), 3840×2160 (large).
4. Lưu JPEG chất lượng 95, bỏ EXIF (gồm cả tọa độ GPS), đặt tên dạng `dataset/site1/small_left.jpg`.
5. Ghi `dataset/manifest.csv` gồm file nguồn, khung cắt và SHA-256 của từng ảnh.

| Site | Ảnh gốc | Khung cắt 16:9 (x0 y0 x1 y1) |
|---|---|---|
| site1 | 4096×3072 | `0 384 4096 2688` |
| site2 | 8160×6144 | `0 777 8160 5367` |
| site3 | 4032×3024 | `0 378 4032 2646` |
| challenge | 4096×2304 | `0 0 4096 2304` |

Kết quả là 24 ảnh (4 cặp × 3 kích thước × 2 ảnh). Ảnh nào cũng đúng kích thước, là JPEG RGB, không còn EXIF, và SHA-256 khớp với manifest.

### Dataset Links

- Thư mục `dataset/` đã chuẩn bị: https://drive.google.com/drive/folders/1HOvnly1k1QHOsV17qLzSAG4ihszNzCqO?usp=sharing
- Ảnh gốc của các thành viên: https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing

Lệnh tạo lại `dataset/` từ ảnh gốc:

```bash
python scripts/prepare_dataset.py --raw raw --out dataset \
    --pair site1=thanh-standard1 --pair site2=tuan-standard2 \
    --pair site3=de-standard2 --pair challenge=chuong-challenge1
```

<a id="gpu-plan"></a>
## 3.7 GPU Execution Plan

### Priorities and Profiling Evidence

Theo bảng timing ở mục 3.5, ba stage chiếm phần lớn thời gian V0 ở cả ba độ phân giải là S2 (keypoint detection), S4 (matching) và S6 (warping and blending). Nhóm ưu tiên ba stage này khi tối ưu ở V2 và V3:

- S4 là hot stage ở small và medium. Thời gian S4 tỷ lệ với $n_A \times n_B$ (tối đa 2000 × 2000 cặp descriptor) và gần như không đổi theo kích thước ảnh. Mỗi cặp descriptor tính độc lập, rất hợp với GPU.
- S2 và S6 tăng theo số pixel. Ở large, S2 là hot stage của site1, S6 là hot stage của site2 và site3. Cả hai đều có phần lớn công việc là phép tính độc lập trên từng pixel.
- S1, S3 và S5 chiếm phần nhỏ hơn, nhưng phần của S1 và S3 tăng theo độ phân giải. Ước lượng Amdahl ở mục 3.5 cho thấy nếu chỉ tăng tốc S2, S4, S6 thì giới hạn speedup giảm khi ảnh lớn lên, nên ba stage còn lại cũng có kernel riêng để không thành phần giới hạn.

### V1 Mapping

V1 dùng kernel đơn giản trên global memory và giữ đúng phương pháp, tham số, thứ tự tính của V0. Dự kiến giữ dữ liệu trên Device giữa các stage để tránh chép qua lại; cách nối interface Host hiện tại của V0 với buffer Device sẽ thống nhất khi cài V1.

| Stage | Một thread, warp hoặc block làm gì | Phần khó song song |
|---|---|---|
| S1 Preprocessing | Một thread cho một pixel ở mỗi lượt: đổi ảnh xám, blur ngang, blur dọc. Trọng số Gaussian trong constant memory | Không đáng kể; stage bị giới hạn bởi băng thông bộ nhớ |
| S2 Detection | Một thread cho một pixel: segment test và score V; một thread cho một pixel: NMS 3×3, ghi ứng viên bằng bộ đếm atomic | Chọn top-N cần sắp xếp toàn cục theo khóa (score, raster); thứ tự output vẫn giống V0 vì khóa không trùng |
| S3 Description | Dùng lại kernel blur của S1 (σ = 2); một thread cho một keypoint tính 256 bit; pattern trong constant memory | Đọc pixel rải rác trong patch; compact keypoint phải giữ thứ tự bằng prefix sum |
| S4 Matching | Một block cho một descriptor A, các thread chia nhau duyệt descriptor B, tính Hamming bằng `__popc`, reduction trong block lấy hai láng giềng gần nhất | Ghi match theo thứ tự indexA bằng prefix sum, không dùng atomic append |
| S5 Homography | Một block cho một hypothesis RANSAC: thread 0 sinh mẫu theo `(seed, id)` và giải DLT, các thread chia nhau tính residual, reduction theo đúng cây cộng của V0 | Điều phối dừng sớm phải duyệt kết quả theo thứ tự ID; solver SVD bằng `double` chạy tuần tự trong block |
| S6 Warp + blend | Một kernel một thread tính H⁻¹ và canvas; một thread cho một pixel canvas: chiếu về B, bilinear, trộn 50/50 với A | Host phải đọc kích thước canvas để cấp phát output; phép tính `double` |

Thời gian mỗi stage trên GPU đo bằng CUDA events, gồm cả các bước phụ (sort, prefix, reduction, điều phối), không chỉ kernel chính. Thời gian chép dữ liệu giữa Host và Device ghi thành cột riêng.

### Optimization Ideas

Mỗi phiên bản sau V1 chỉ thêm một ý tưởng chính và đo riêng tác dụng của nó. Danh sách cuối chốt sau khi profiling V1; ba ý tưởng nhóm dự kiến thử trước, theo ba stage ưu tiên:

1. S4: dùng lại descriptor B qua shared memory. Một block xử lý nhiều descriptor A, nạp từng tile descriptor B vào shared memory một lần cho mọi A trong block, thay vì mỗi block đọc lại toàn bộ B từ global memory. Có thể thử thêm một warp cho một descriptor A, dùng shuffle để lấy hai láng giềng gần nhất.
2. S2: shared memory tile và gộp score với NMS. Mỗi block nạp tile ảnh cùng halo vào shared memory, tính score cho cả vùng halo rồi làm NMS ngay trong kernel, bỏ một lượt ghi và đọc lại bản đồ score. Thay sắp xếp toàn bộ ứng viên bằng chọn top-N theo histogram score.
3. S6: một kernel ghi mỗi pixel canvas đúng một lần. Gộp xóa canvas, chép A và gather B vào một kernel; chỉ chiếu và nội suy trong bounding box của B; tính trước các số hạng chỉ phụ thuộc hàng của phép chiếu để giảm phép tính `double`.

Ở mức pipeline, nhóm cũng xem xét xử lý ảnh A và B trên hai CUDA stream ở S1 đến S3 và dùng pinned memory khi chép ảnh, nếu profiling cho thấy phần truyền dữ liệu đáng kể.

### Risks

- S5 và S6 dùng `double`. T4 có throughput phép tính 64 bit thấp hơn nhiều so với 32 bit, nên hai stage này có thể tăng tốc ít hơn dự kiến. Đổi sang `float` là đổi phương pháp, phải áp dụng cả cho V0 và cả Nhóm quyết định.
- Giữ output giống V0: thứ tự keypoint của S2, thứ tự match và cách xử lý hai khoảng cách bằng nhau ở S4, điểm dừng RANSAC ở S5 đều phải giống hệt V0. Sort, atomic và reduction song song dễ làm đổi thứ tự nếu không có khóa xác định.
- Chi phí truyền dữ liệu: ở large, mỗi ảnh vào và nhất là panorama đều lớn, thời gian chép giữa Host và Device có thể vượt thời gian kernel. Nếu chỉ báo thời gian kernel, speedup end-to-end sẽ bị đánh giá cao hơn thực tế.
- S5 có ít match (vài trăm) nên mỗi block có ít việc, và phần giải DLT chạy tuần tự; S5 có thể không nhanh hơn V0 nhiều.

### V1 Verification

Mỗi stage V1 được so với V0 trên cùng input: S2 và S3 so keypoint, descriptor từng bit và đúng thứ tự; S4 so danh sách match; S5 so điểm dừng, mask inlier và H qua sai số chiếu; S1 và S6 so ảnh với sai lệch cho phép đã ghi trong tài liệu stage. Sau đó chạy toàn pipeline trên bộ cặp synthetic với cùng ngưỡng sai số homography (RMSE ≤ 0.5 px, Max ≤ 1 px) và trên dataset ở cả ba độ phân giải.

<a id="references"></a>
## References

### S1 - Tuan


| Nguồn | Ngày truy cập | Dùng cho phần nào |
|---|---|---|
| Đề cuối kỳ (đề bài môn học), mục 2.1, 2.2 và phần tối ưu ở mức pipeline | 07/10, 21/10 | Mô tả S1; ý tưởng gộp grayscale với blur, pinned memory, CUDA stream ở mục 6.3 tài liệu S1 |
| [ITU-R Recommendation BT.601-7 (03/2011)](https://www.itu.int/rec/R-REC-BT.601) | 21/10 | Hệ số đổi ảnh xám 0.299, 0.587, 0.114 |
| [NVIDIA CUDA C++ Best Practices Guide 13.0](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html), mục 9.1.2, 10.1.1, 10.1.2, 10.2.1, 10.2.3, 10.2.6, 11.3 | 21/10 | Mục 6 tài liệu S1: CUDA events, pinned memory, chồng truyền dữ liệu với tính toán, coalescing, shared memory, constant memory, kích thước block |
| Benchmark V0 ngày 20/10 của Thành: `results/p3-t04/benchmark/summary.csv` trên nhánh `thanh/p3-results` | 21/10 | Thời gian và tỷ trọng S1 ở mục 6.1 tài liệu S1 |

### S4/S5 - De


| Nguồn | Dùng cho phần nào | Phạm vi đã đọc/kiểm chứng |
|---|---|---|
| Đề cuối kỳ (đề bài môn học), đề giữa kỳ (đề bài môn học) | Yêu cầu pipeline, sequential, GPU, fairness, synthetic và báo cáo | Đọc bản Markdown local; đã đối chiếu yêu cầu giữa kỳ với PDF gốc khi hoàn thiện P4 |
| Kế hoạch Nhóm và bảng công việc trong repo; contract tại `src/common.h` của bài nộp | Phạm vi task, thư viện, quy ước dữ liệu và nghiệm thu | Đọc tài liệu và contract hiện tại; trạng thái công việc được quản lý trong repo |
| [Lowe, D. G. (2004). Distinctive Image Features from Scale-Invariant Keypoints](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), IJCV 60(2), 91–110 | S4: ý tưởng ratio test, nguồn ngưỡng khảo sát | Đọc mục 7.1–7.2 qua PDF tác giả; thí nghiệm dùng SIFT, không chuyển kết luận chất lượng sang binary descriptor |
| [OpenCV 4.13.0: Feature Matching](https://docs.opencv.org/4.13.0/dc/dc3/tutorial_py_matcher.html) | S4: Hamming/L2, exact search, ratio và cross-check | Đọc tài liệu chính thức; không thêm OpenCV vào chương trình |
| [OpenCV 4.13.0: BFMatcher, create](https://docs.opencv.org/4.13.0/d3/da1/classcv_1_1BFMatcher.html) | S4: Hamming cho ORB/BRISK/BRIEF; cross-check là alternative cho ratio, không bắt buộc | Đọc trực tiếp mục create; phân biệt Hamming và Hamming2 cho ORB WTA_K=3/4 |
| [OpenCV 4.13.0: USAC](https://docs.opencv.org/4.13.0/de/d3e/tutorial_usac.html) | S5: uniform sampling, MSAC score, reprojection error, homography collinearity, standard stopping và all-inlier fitting | Đọc trực tiếp tutorial; không đồng nhất minimal solver hoặc mọi policy của OpenCV với normalized DLT/prefix deterministic của Nhóm |
| [OpenCV 4.13.0: Feature Matching with FLANN](https://docs.opencv.org/4.13.0/d5/d6f/tutorial_feature_flann_matcher.html) | S4: hướng approximate nearest-neighbor và binary matching | Đọc tài liệu chính thức; không dùng implementation làm dependency |
| [OpenCV: lsh_index.h](https://github.com/opencv/opencv/blob/4.x/modules/flann/include/opencv2/flann/lsh_index.h), nhánh 4.x | S4: ví dụ LSH index cho phương án ANN | Kiểm tra nguồn chính thức cho LSH index; chưa khảo sát đầy đủ implementation và không chép code |
| [Lindenberger, P., Sarlin, P.-E., Pollefeys, M. (2023). LightGlue: Local Feature Matching at Light Speed](https://openaccess.thecvf.com/content/ICCV2023/html/Lindenberger_LightGlue_Local_Feature_Matching_at_Light_Speed_ICCV_2023_paper.html), ICCV | S4: learned sparse matcher | Metadata/abstract từ CVF qua kết quả tìm kiếm; không benchmark tại Nhóm |
| [Wang, Y. et al. (2024). Efficient LoFTR: Semi-Dense Local Feature Matching with Sparse-Like Speed](https://openaccess.thecvf.com/content/CVPR2024/html/Wang_Efficient_LoFTR_Semi-Dense_Local_Feature_Matching_with_Sparse-Like_Speed_CVPR_2024_paper.html), CVPR | S4: learned semi-dense matching | Metadata/abstract và đoạn phương pháp từ PDF được CVF lập chỉ mục |
| [Li, X., Rao, T., Pan, C. (2025). EDM: Efficient Deep Feature Matching](https://openaccess.thecvf.com/content/ICCV2025/html/Li_EDM_Efficient_Deep_Feature_Matching_ICCV_2025_paper.html), ICCV | S4: detector-free matching gần đây | Metadata/abstract từ CVF qua kết quả tìm kiếm |
| [Edstedt, J. et al. RoMa v2: Harder Better Faster Denser Feature Matching](https://arxiv.org/abs/2511.15706), preprint 2025, v3 cập nhật 06/07/2026 | S4: learned dense matching gần đây | Đọc trang arXiv/abstract; truy cập bản HTML, không tuyên bố đã đọc trọn paper |
| [Lee, J., Kang, S., Yoo, S. (2026). MV-RoMa: From Pairwise Matching into Multi-View Track Reconstruction](https://arxiv.org/abs/2603.27542), preprint | S4: phân biệt hai ảnh và nhiều view | Đọc metadata/abstract arXiv |
| [Brown, M., Lowe, D. G. (2007). Automatic Panoramic Image Stitching using Invariant Features](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf), IJCV 74, 59–73 | S5: RANSAC với mẫu 4 correspondence, DLT và consensus | Đọc mục 3.1 qua PDF tác giả; không dùng tham số paper như cấu hình đã chốt của Nhóm |
| [Harker, M., O'Leary, P. (2005). Computation of Homographies](https://www.robots.ox.ac.uk/~phst/BMVC2005/papers/190/paper190.pdf), BMVC | S5: centering/RMS normalization và homogeneous DLT | Đọc mục 2.3 và đoạn DLT/nullspace; không chọn phương pháp residualization cải tiến của paper |
| [Drmac, Z., Veselic, K. New Fast and Accurate Jacobi SVD Algorithm: II, LAPACK Working Note 170](https://www.netlib.org/lapack/lawnspdf/lawn170.pdf) | S5: cơ sở one-sided Jacobi SVD và stopping theo orthogonality | Đọc phần giới thiệu và cyclic rotation/stopping; tự cài core unpreconditioned cho 9 cột, không sao chép/link LAPACK hoặc áp dụng toàn bộ preconditioned algorithm của paper |
| [Vigna, S. (2015). SplitMix64 reference](https://prng.di.unimi.it/splitmix64.c) | S5: mixer/next-state trong mã giả RNG deterministic | Đọc toàn bộ nguồn reference/public-domain notice; stream initialization và bounded sampling là policy của Nhóm |
| [Fowler/Noll/Vo: FNV Hash](https://www.isthe.com/chongo/tech/comp/fnv/index.html) | S5: fingerprint chẩn đoán thứ tự sample | Đọc công thức FNV-1a và hằng số 64-bit; biến thể local XOR toàn bộ chỉ số rồi nhân, không phải FNV-1a trên byte stream và không tham gia chọn homography |
| [Chum, O., Matas, J. (2005). Matching with PROSAC – Progressive Sample Consensus](https://cmp.felk.cvut.cz/~matas/papers/chum-prosac-cvpr05.pdf), CVPR | S5: phương án sampling dựa trên ranking | Đọc abstract và mục thuật toán qua PDF tác giả |
| [Barath, D. et al. (2020). MAGSAC++, a Fast, Reliable and Accurate Robust Estimator](https://openaccess.thecvf.com/content_CVPR_2020/html/Barath_MAGSAC_a_Fast_Reliable_and_Accurate_Robust_Estimator_CVPR_2020_paper.html), CVPR | S5: robust estimation và weighted refinement | Abstract và đoạn phương pháp/thí nghiệm từ CVF được tìm kiếm trả về; mở trang trực tiếp gặp lỗi |
| [Ivashechkin, M., Barath, D., Matas, J. (2021). VSAC: Efficient and Accurate Estimator for H and F](https://openaccess.thecvf.com/content/ICCV2021/html/Ivashechkin_VSAC_Efficient_and_Accurate_Estimator_for_H_and_F_ICCV_2021_paper.html), ICCV | S5: alternative RANSAC-type không chọn | Metadata/abstract và đoạn mô tả verification/failure từ CVF được tìm kiếm trả về; không chuyển số đo paper sang Nhóm |

Tra cứu có mục tiêu, không phải khảo sát đầy đủ mọi công bố đến năm 2026. Các đánh giá về mức phù hợp với C++/CUDA của đồ án là suy luận từ yêu cầu và contract hiện có. Mọi kết luận chất lượng/hiệu năng trên dataset Nhóm cần kết quả chạy thực tế.

#### GPU Mapping References

Đế bổ sung ngày 21/10/2026, dùng cho mục 6 của S4/S5 và nội dung GPU trong báo cáo. Đây là nguồn thiết kế, không cung cấp số đo speedup của nhóm.

| Nguồn | Dùng cho phần nào | Phạm vi đã đọc/kiểm chứng |
|---|---|---|
| [NVIDIA CUDA Math API 13.0 — Integer Intrinsics](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-math-api/cuda_math_api/group__CUDA__MATH__INTRINSIC__INT.html) | S4: `__popc` trên word 32 bit ở Device | Mục khai báo và ý nghĩa `__popc` |
| [NVIDIA CUDA C++ Best Practices Guide 13.0](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html) | S4/S5: profiling, bố trí bộ nhớ, truyền Host–Device, timing và giới hạn Amdahl | Các mục Assess, Timing, Memory Optimizations và Data Transfer; mapping block/thread và prefix deterministic là thiết kế của nhóm |
| Benchmark V0 scalar ngày 20/10/2026: `results/p3-t04/benchmark/summary.csv`, CSV/log từng site và `environment.json` | Căn cứ hiệu năng cho mapping S4/S5 | Đối chiếu chín hàng average với 45 lần đo gốc; các file kết quả được lưu riêng, không commit vào repo |

### S2/S3 - Thanh


| Nguồn | Dùng cho phần nào | Phạm vi đã đọc/kiểm chứng |
|---|---|---|
| [Rosten, E., Drummond, T. (2006). Machine learning for high-speed corner detection](https://www.edwardrosten.com/work/rosten_2006_machine.pdf), ECCV | S2: segment test, vòng tròn 16 điểm, score V | Mục segment test và score V; không dùng cây quyết định học máy |
| [Rosten, E., Porter, R., Drummond, T. (2010). Faster and better: a machine learning approach to corner detection](https://arxiv.org/abs/0810.2434), IEEE TPAMI | S2: so sánh độ lặp lại theo số điểm liên tiếp n | Trang arXiv và phần đánh giá độ lặp lại |
| [Calonder, M., Lepetit, V., Strecha, C., Fua, P. (2010). BRIEF: Binary Robust Independent Elementary Features](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf), ECCV | S3: test nhị phân, làm mượt σ = 2 trên cửa sổ 9×9, cách lấy mẫu G II | Đọc mục 3.1-3.3 |
| [Rublee, E., Rabaud, V., Konolige, K., Bradski, G. (2011). ORB: an efficient alternative to SIFT or SURF](https://doi.org/10.1109/ICCV.2011.6126544), ICCV | S2/S3: phương án xếp hạng Harris và rotated BRIEF, patch 31×31 | Dùng cho phần phương án đã cân nhắc |
| [NVIDIA CUDA C++ Best Practices Guide 13.0](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html) | S2/S3 mục 6: CUDA events, coalescing, shared memory, constant memory, kích thước block | Mục 9.1.2, 10.2.1, 10.2.3, 10.2.6, 11.3; truy cập 21/10/2026 |
| [NVIDIA CUDA Programming Guide, mục 5.4.6.2 Warp Vote Functions](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-extensions.html#warp-vote-functions) | S3 mục 6.3: ghép bit descriptor bằng `__ballot_sync` | Mô tả `__ballot_sync`; truy cập 21/10/2026 |
| Benchmark V0 ngày 20/10/2026: `results/p3-t04/benchmark/summary.csv` và log `--debug` trên nhánh `thanh/p3-results` | S2/S3 mục 6.1, báo cáo 3.5 timing | Hàng average, minimum và số keypoint, match, canvas trong log |
| [OpenCV 4.x, `features2d.hpp`](https://github.com/opencv/opencv/blob/4.x/modules/features2d/include/opencv2/features2d.hpp) và [`orb.cpp`](https://github.com/opencv/opencv/blob/4.x/modules/features2d/src/orb.cpp) | S2: ngưỡng FAST mặc định 10 (FAST) và 20 (ORB); S3: ORB làm mượt Gaussian 7×7, σ = 2 | Đọc khai báo `create` và dòng `GaussianBlur`; không thêm OpenCV vào chương trình |
| [Vigna, S. SplitMix64 reference](https://prng.di.unimi.it/splitmix64.c) | S3: bộ sinh số ngẫu nhiên cho pattern | Chuyển sang Python trong công cụ sinh pattern |

### S6 - Chuong


| Nguồn | Dùng cho phần nào | Phạm vi đã đọc/kiểm chứng |
|---|---|---|
| [Szeliski, R. Image alignment and stitching: A tutorial](https://doi.org/10.1561/0600000009), Foundations and Trends in Computer Graphics and Vision 2(1) | S6: hệ quy chiếu, resampling, compositing | Mục 6 bản technical report; truy cập 12/10/2026 |
| [Brown, M., Lowe, D. G. (2007). Automatic panoramic image stitching using invariant features](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf), IJCV 74 | S6: so sánh linear blending và multi-band, giới hạn khi có vật chuyển động | Mục 6–7 và 9; truy cập 12/10/2026 |
| [Burt, P. J., Adelson, E. H. (1983). A multiresolution spline with application to image mosaics](https://doi.org/10.1145/245.247), ACM TOG 2(4) | S6: nguồn gốc multi-band blending (phương án không chọn) | Tên và abstract; chưa đọc toàn văn |
| [NVIDIA CUDA C Programming Guide 12.5, texture fetching](https://docs.nvidia.com/cuda/archive/12.5.0/cuda-c-programming-guide/index.html#texture-fetching) | S6 mục 6: giới hạn độ chính xác của bilinear phần cứng | Mục 15.1–15.2; truy cập 12/10/2026 |
| [NVIDIA CUDA C++ Best Practices Guide 13.0](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html) | S6 mục 6: throughput phép tính 64 bit trên compute capability 7.5, pinned memory, CUDA events | Mục 12.1.1 (bảng 5), 10.1.1–10.1.2, 9.1.2; truy cập 21/10/2026 |
| Benchmark V0 ngày 20/10/2026 (`results/p3-t04/` trên nhánh `thanh/p3-results`) và lần Run all notebook ngày 21/10/2026 | S6 mục 6.1, báo cáo 3.5 panorama | Thời gian S6, kích thước canvas, panorama và crop |

### AI Tool Usage

Nhóm sử dụng ChatGPT/Codex để hỗ trợ giải thích, cài đặt, kiểm lỗi, tổ chức notebook và rà báo cáo. Thành viên chịu trách nhiệm hiểu, kiểm chứng và giải thích mã nguồn cùng kết quả. Nguồn thuật toán và code ngoài được ghi tại chỗ sử dụng, trong bảng trên và đầu các file nguồn liên quan.

<a id="checklist"></a>
## Reproducibility Checklist

Cell cuối tổng hợp bằng chứng của phiên và in `READY` khi mọi kiểm tra đạt, gồm việc đã xem panorama/crop và đối chiếu nội dung báo cáo. Hai cờ xác nhận phản ánh việc rà bản này; không yêu cầu các thành viên xác nhận lại. Không có bước bắt buộc bật `REQUIRE_COMPLETE_RUN=True`.

Sau khi rà, lưu notebook có đầy đủ output để nộp. Bản trong Git giữ cùng source cell, với `execution_count=null` và `outputs=[]`.

In [ ]:
CHECKS["source_unchanged"] = source_manifest(PROJECT) == SOURCE_HASHES
CHECKS["quality_review"] = QUALITY_REVIEW_CONFIRMED
CHECKS["final_report"] = FINAL_REPORT_CONFIRMED
display(pd.DataFrame([{"check": name, "passed": bool(value)} for name, value in CHECKS.items()]))
ready = all(CHECKS.values())
receipt = {"session_id": SESSION_ID, "status": "READY" if ready else "INCOMPLETE",
           "checks": CHECKS, "accelerated_stages": ACCELERATED_STAGES,
           "raw_synthetic_sha256": raw_digest,
           "evidence_files": sorted(str(path.relative_to(RESULTS)) for path in RESULTS.rglob("*") if path.is_file())}
(RESULTS / "run_receipt.json").write_text(json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8")
print("Run status:", receipt["status"])
print("Evidence directory:", RESULTS)
if REQUIRE_COMPLETE_RUN and not ready:
    raise RuntimeError("Final submission gate failed: complete all pending evidence and report checks.")